<a href="https://colab.research.google.com/github/erdemustun/Dosyalar/blob/master/HisseKurumSeriDurumlar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bazı aracı kurumları takip etmek için yapıldı. Bofa_2107.xlsx ,Isyat_2207.xlsx şeklinde Kurum Hisse Dağılımı/İşlem Özeti matriksden excel aktarılır,hangi kurumu takip etmek istiyorsak ve tek günlük olacak.HisseKurumSeriDurumlar.xlsx ilk kez oluşturdaktan sonra sürekli eklenebilir,günlük olarak
Fiili dolaşım verilerinide aracı kurumla birlikte güncelle

In [ ]:
#10 dosya
# AK YATIRIM. Ak_ddmm
# BANK-OF-AMERICA YATIRIM BANK Bofa_ddmm
# DENIZ YATIRIM MENKUL Deniz_ddmm
# GEDIK YATIRIM Gedik_ddmm
# HSBC YATIRIM  Hsbc_ddmm
# IS YATIRIM  Isyat_ddmm
# Marbas YATIRIM. Marbas_ddmm
# QNB YATIRIM MENKUL  Qnb_ddmm
# TERA YATIRIM MENKUL Tera_ddmm
# YAPI KREDI YAT. Ykb_ddmm

#Günlük pgc(prime dan 20 kurum ve yalnızca hisseler seçilecek ve dosya adı pgc_ddmm.xlsx)

In [ ]:
!pip install git+https://github.com/rongardF/tvdatafeed tradingview-screener
!pip install retrying
#!pip install openpyxl
!pip install networkx

In [ ]:
import numpy as np
import pandas as pd
from tvDatafeed import TvDatafeed, Interval
tv = TvDatafeed()
import warnings
from tradingview_screener import Query, Column
warnings.simplefilter(action='ignore')
from datetime import datetime
import traceback
from google.colab import files
from retrying import retry
import sys
import os
from glob import glob
from collections import defaultdict
import re
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, numbers
import unicodedata
import networkx as nx
import shutil #zip işlemleri için

In [ ]:
@retry(wait_fixed=4000, stop_max_attempt_number=3) # Hata olması halinde 4 sn bekleyip tekrar deneyecek, 3 kez deneyecek
def veri_getir(symbol,intervald):
    return tv.get_hist(symbol=symbol, exchange='BIST', interval=intervald, n_bars=30)

In [ ]:
# günlük / haftalık mı çalışacak onun için
dt = datetime.now()
gun = dt.isoweekday()

if gun < 8:
    periyot = 'G'
elif gun == 8:
    periyot = 'H'
else:
    print("[Pazartesi-Cuma] ve Pazar günleri çalışır")
    sys.exit()

if periyot=='H':
    period='7y'
    intervald=Interval.in_weekly
    dosya='Haftalik'
    freqs = 'W'
    offset = pd.DateOffset(weeks=1)  # Haftalık frekans için bir hafta eklenir
elif periyot=='G':
    period='2y'
    intervald=Interval.in_daily
    dosya='Gunluk'
    freqs = 'D'
    offset = pd.DateOffset(days=1)  # Günlük frekans için bir gün eklenir
elif periyot=='S':
    period='1y'
    intervald=Interval.in_1_hour
    dosya='Saatlik'
elif periyot=='D':
    period='15d'
    intervald=Interval.in_15_minute
    dosya='15dakikalık'
else:
    print('Hatalı girdi')

In [ ]:
piyasa= 'turkey'
#piyasa= 'germany'
#piyasa= 'america'
options = ['stock']  #['stock', 'fund','dr']

#Burada tupple dönüyor
HisseAd_Sorgu = (Query().set_markets(f'{piyasa}')
                .select('name', 'type')
                .where(Column('type').isin(options) #(['stock', 'fund'])
                    )
                .limit(20000) #deault değeri 50 olduğu için hepsini getirmiyor.Onu aşmak için yüksek değer koydum.
                .order_by('name')
                .get_scanner_data())[1]

# DataFrame'e dönüştür
HisseAd = pd.DataFrame(HisseAd_Sorgu, columns=['ticker', 'name', 'type'])
#len(HisseAd)
#HisseAd.to_csv('HisseAd.csv')

# Piyasa bazında filtreleme
if piyasa == 'america':
    # Amerika için NYSE veya NASDAQ ile başlayanları seç
    filtered_HisseAd = HisseAd[HisseAd['ticker'].str.startswith(('NYSE', 'NASDAQ'))]
elif piyasa == 'turkey':
    # Turkey için herhangi bir filtreleme uygulanmaz
    filtered_HisseAd = HisseAd
elif piyasa == 'germany':
    # Germany için özel filtreleme (örneğin XETRA ile başlayanlar)
    filtered_HisseAd = HisseAd[HisseAd['ticker'].str.startswith(('GETTEX', 'TRADEGATE', 'XETRA'))]
else:
    # Diğer piyasalar için de benzer filtrelemeler eklenebilir
    filtered_HisseAd = HisseAd

# Listeye dönüştür
Hisseler = filtered_HisseAd['ticker'].tolist()

In [ ]:
# # Eksik çekilenleri bulmak için

# Hisseler #hisselerin listesi
# tum_veriler_hisseler = tum_veriler['Hisse'].unique()  # Bu sizin gerçek veriniz

# # Hisseler listesini temizle
# hisseler_temiz = [h.replace('BIST:', '') for h in Hisseler]

# # Eksik hisseleri bul
# eksik_hisseler = list(set(hisseler_temiz) - set(tum_veriler_hisseler))

# print(f"Toplam eksik hisse: {len(eksik_hisseler)}")
# print("Eksik hisseler:", eksik_hisseler)

In [ ]:
def veri_indir_ve_eksikleri_tamamla(Hisseler, intervald, veri_getir_fonksiyonu):

    # Hisse verilerini indirir, eksikleri bulur ve tekrar dener.

    def hisse_verisi_hazirla(hisse_tam, hisse_temiz, veri):
        """Veriyi hazırlayan yardımcı fonksiyon"""
        try:
            data = veri.copy()
            data = data[["datetime", "close"]].rename(columns={"datetime": "tarih"})
            data["Hisse"] = hisse_temiz
            data["tarih"] = pd.to_datetime(data["tarih"].dt.strftime("%d/%m/%Y"), dayfirst=True)
            data["degisim"] = data["close"].pct_change() * 100
            return data
        except Exception as e:
            print(f"  ⚠️ Veri hazırlama hatası ({hisse_temiz}): {e}")
            return None

    # Ana veri DataFrame'i
    tum_veriler = pd.DataFrame(columns=["tarih", "Hisse", "close", "degisim"])

    # 1. TÜM HİSSELERİ İNDİR
    toplamHisse = len(Hisseler)
    print(f"📥 Toplam {toplamHisse} hisse indirilecek...")

    basarisiz_hisseler = []

    for i, hisse in enumerate(Hisseler, 1):
        try:
            print(f"\r🔄 İşleniyor: {hisse} ({i}/{toplamHisse})", end='')

            data = veri_getir_fonksiyonu(hisse, intervald).reset_index()
            if data is None or data.empty:
                basarisiz_hisseler.append(hisse)
                continue


            # Veriyi hazırla
            hisse_temiz = hisse.replace("BIST:", "")
            hazir_veri = hisse_verisi_hazirla(hisse, hisse_temiz, data)

            if hazir_veri is not None:
                tum_veriler = pd.concat([tum_veriler, hazir_veri], ignore_index=True)

        except Exception as e:
            basarisiz_hisseler.append(hisse)
            continue

    # İlk veri temizliği
    tum_veriler['degisim'] = tum_veriler['degisim'].fillna(0)
    tum_veriler = tum_veriler.dropna(subset=['tarih'])
    tum_veriler['tarih'] = pd.to_datetime(tum_veriler['tarih'])
    tum_veriler['degisim'] = tum_veriler['degisim'].astype(float)

    print(f"\n\n✅ İlk indirme tamamlandı. {tum_veriler['Hisse'].nunique()} hisse indirildi.")

    # 2. EKSİK HİSSELERİ BUL
    hisseler_temiz = [h.replace("BIST:", "") for h in Hisseler]
    mevcut_hisseler = tum_veriler['Hisse'].unique().tolist()

    eksik_hisseler = [h for h in hisseler_temiz if h not in mevcut_hisseler]

    if eksik_hisseler:
        print(f"\n🔍 {len(eksik_hisseler)} eksik hisse bulundu: {eksik_hisseler}")

        # 3. EKSİK HİSSELERİ YENİDEN DENE
        print(f"\n🔄 Eksik hisseler yeniden indiriliyor...")

        for i, eksik in enumerate(eksik_hisseler, 1):
            hisse_tam = f"BIST:{eksik}"

            try:
                print(f"\r📥 {eksik} ({i}/{len(eksik_hisseler)})", end='')

                data = veri_getir_fonksiyonu(hisse_tam, intervald).reset_index()

                if data is not None and not data.empty:
                    hazir_veri = hisse_verisi_hazirla(hisse_tam, eksik, data)

                    if hazir_veri is not None:
                        tum_veriler = pd.concat([tum_veriler, hazir_veri], ignore_index=True)
                        print(f" ✅", end='')
                        continue

                print(f" ❌", end='')

            except Exception as e:
                print(f" ❌", end='')
                continue

    # Son kontrol
    son_hisseler = tum_veriler['Hisse'].unique().tolist()
    hala_eksik = [h for h in hisseler_temiz if h not in son_hisseler]

    return tum_veriler, hala_eksik

# KULLANIM
tum_veriler, hala_eksik_olanlar = veri_indir_ve_eksikleri_tamamla( Hisseler,intervald, veri_getir)  # sizin veri_getir fonksiyonunuz

# Sonuç raporu
print("\n" + "="*60)
print("📊 İNDİRME RAPORU")
print("="*60)
print(f"Toplam veri satırı: {len(tum_veriler):,}")
print(f"Benzersiz hisse sayısı: {tum_veriler['Hisse'].nunique()}")
print(f"Tarih aralığı: {tum_veriler['tarih'].min().date()} - {tum_veriler['tarih'].max().date()}")

if hala_eksik_olanlar:
    print(f"\n⚠️ HALA EKSİK OLAN HİSSELER ({len(hala_eksik_olanlar)}):")
    for hisse in hala_eksik_olanlar[:20]:  # İlk 20'yi göster
        print(f"  - {hisse}")

    if len(hala_eksik_olanlar) > 20:
        print(f"  ... ve {len(hala_eksik_olanlar) - 20} hisse daha")
else:
    print("\n✅ TÜM HİSSELER BAŞARIYLA İNDİRİLDİ!")


In [ ]:
# tum_veriler[tum_veriler["Hisse"] == "CMBTN"]

In [ ]:
# #Eski kayıtlar niye geliyor için bakıldı
# esit_tarih_kayitlari = tum_veriler[tum_veriler['tarih'] == pd.to_datetime(tum_veriler['tarih'].min().strftime('%d/%m/%Y'), dayfirst=True)]
# display(esit_tarih_kayitlari)

# Fiili Dolaşımı webden direkt almak için

In [ ]:
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def clean_number(x):
    """ '1.234.567,89' → 1234567.89 """
    if not isinstance(x, str):
        return x
    x = x.replace(".", "").replace(",", ".")
    try:
        return float(x)
    except:
        return None

url = "https://kap.org.tr/tr/tumKalemler/kpy41_acc5_fiili_dolasimdaki_pay"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "tr-TR,tr;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://kap.org.tr/",
    "Connection": "keep-alive",
}

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session.mount("https://", HTTPAdapter(max_retries=retries))


@retry(wait_fixed=5000, stop_max_attempt_number=5)
def get_kap_soup():
    r = session.get(url, headers=headers, timeout=(10, 30))
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

# soup = get_kap_soup(url, headers)
soup = get_kap_soup()
table = soup.find("table")

if table is None or not table.find("tbody"):
    raise RuntimeError("❌ Tablo HTML içinde yok (KAP boş sayfa döndürmüş olabilir)")

tbody = table.find("tbody")

if tbody is None:
    print("tbody bulunamadı!")
    exit()

rows = []

for tr in tbody.find_all("tr"):
    tds = tr.find_all("td")

    # Son 3 sütunu al (borsa kodu, pay adedi, oran)
    if len(tds) >= 3:
        borsa_kodu = tds[-3].get_text(strip=True)
        pay_adedi = tds[-2].get_text(strip=True)
        oran = tds[-1].get_text(strip=True)

        if borsa_kodu:  # Boş değilse ekle
            rows.append([borsa_kodu, pay_adedi, oran])

# Önce tüm veriyi DataFrame'e al
FiiliDolasim_df_raw = pd.DataFrame(rows, columns=[
    "Borsa Kodu",
    "Fiili Dolaşımdaki Pay Adedi",
    "Fiili Pay/Sermaye Oranı (%)"
])

# Sayısal dönüşümler
FiiliDolasim_df_raw["Fiili Dolaşımdaki Pay Adedi"] = FiiliDolasim_df_raw["Fiili Dolaşımdaki Pay Adedi"].apply(clean_number)
FiiliDolasim_df_raw["Fiili Pay/Sermaye Oranı (%)"] = FiiliDolasim_df_raw["Fiili Pay/Sermaye Oranı (%)"].apply(clean_number)

print(f"Toplam {len(FiiliDolasim_df_raw)} satır çekildi.\n")

# Duplicate'leri kontrol et ve göster
duplicates = FiiliDolasim_df_raw[FiiliDolasim_df_raw.duplicated(subset=["Borsa Kodu"], keep=False)]
if len(duplicates) > 0:
    print(f"⚠️ Aynı borsa koduna sahip {len(duplicates)} satır bulundu:")
    print(duplicates.sort_values("Borsa Kodu"))
    print("\n" + "="*50)
    print("En son satırlar tutulacak...\n")
else:
    print("✓ Tüm borsa kodları zaten tekil.\n")

# Aynı borsa kodundan en SONUNCUsunu tut (keep='last')
FiiliDolasim_df = FiiliDolasim_df_raw.drop_duplicates(subset=["Borsa Kodu"], keep="last")

# Borsa koduna göre sırala
FiiliDolasim_df = FiiliDolasim_df.sort_values("Borsa Kodu").reset_index(drop=True)

# Excel'e kaydet
FiiliDolasim_df.to_excel("Fiili_Dolasim.xlsx", index=False)

# print(f"✓ Nihai DataFrame: {len(FiiliDolasim_df)} tekil borsa kodu")
# print("\nİlk 20 satır:")
# print(FiiliDolasim_df.head(20))

# Para Giriş Çıkış düzenlemek,kurumların hepsini oradan almak için

In [ ]:
class PGCProcessor:
    """ParaGirişÇıkış dosyalarını işlemek için merkezi sınıf"""

    # Sabitler
    MAX_SYMBOL_LENGTH = 6
    DETAIL_HEADERS = ['Alanlar', 'Net Hacim', 'Yüzde', 'Net Adet', 'Satanlar']

    # Yeni formattan eski formata kolon eşleştirmesi
    COLUMN_MAPPING = {
        'symbol': 0,           # Sembol
        'para_gc': 1,          # Para G.Ç.
        'alis_hacim': 2,       # Alış N.Hacim
        'alis_ort': 3,         # Alış Ort.
        'alis_yuzde': 4,       # Alış %
        'satis_hacim': 5,      # Satış N.Hacim
        'satis_ort': 6,        # Satış Ort.
        'satis_yuzde': 7       # Satış %
    }

    # Çıktı kolon isimleri
    SUMMARY_COLUMNS = [
        'Sembol', 'Alanlar', 'Alici_Net_Hacim', 'Alici_Ortalama',
        'Alici_Yuzdesi', 'Alici_Net_Lot', 'Satanlar', 'Satici_Net_Hacim',
        'Satici_Ortalama', 'Satici_Yuzdesi', 'Satici_Net_Lot', 'Para_Giris_Cikis'
    ]

    DETAIL_COLUMNS = [
        'Sembol', 'Alanlar', 'Alici_Net_Hacim', 'Alici_Yuzdesi',
        'Alici_Net_Lot', 'Satanlar', 'Satici_Net_Hacim',
        'Satici_Yuzdesi', 'Satici_Net_Lot'
    ]

    @staticmethod
    def normalize_text(text):
        """Türkçe karakterleri normalize et"""
        if pd.isna(text):
            return ""
        text = str(text).strip().upper()
        text = unicodedata.normalize("NFD", text)
        return "".join(c for c in text if unicodedata.category(c) != "Mn")

    def __init__(self, df):
        """DataFrame'i işleme hazırla"""
        self.df = df.copy()
        self.symbol_col = df.columns[0] if 'Sembol' not in df.columns else 'Sembol'
        self._mark_rows()
        self._fill_symbols()

    def _mark_rows(self):
        """Satır tiplerini işaretle"""
        self.df['_is_symbol_row'] = False
        self.df['_is_detail_header'] = False

        # Sembol satırlarını bul
        symbol_mask = (
            self.df[self.symbol_col].notna() &
            (self.df[self.symbol_col].astype(str).str.strip() != '')
        )

        for idx in self.df[symbol_mask].index:
            self.df.loc[idx, '_is_symbol_row'] = True

            # Detay başlığını işaretle
            next_idx = idx + 1
            if next_idx in self.df.index and len(self.df.columns) > 1:
                second_col_val = str(self.df.iloc[next_idx, 1]).strip()
                if second_col_val in self.DETAIL_HEADERS:
                    self.df.loc[next_idx, '_is_detail_header'] = True

    def _fill_symbols(self):
        """Sembolleri forward fill ile doldur"""
        self.df[self.symbol_col] = self.df[self.symbol_col].ffill()

    def get_summary_rows(self):
        """Özet satırlarını eski formatta döndür"""
        # Sembol satırlarını filtrele
        symbol_df = self.df[
            (self.df['_is_symbol_row'] == True) &
            (self.df[self.symbol_col].astype(str).str.len() < self.MAX_SYMBOL_LENGTH)
        ].copy()

        # Yardımcı sütunları kaldır
        symbol_df = self._remove_helper_columns(symbol_df)

        # Boş sütunları temizle
        symbol_df = symbol_df.dropna(axis=1, how='all')

        # Yeni formattan eski formata dönüştür
        return self._convert_to_old_format_summary(symbol_df)

    def get_detail_rows(self):
        """Detay satırlarını eski formatta döndür"""
        # Detay satırlarını filtrele
        detail_df = self.df[
            (self.df['_is_symbol_row'] == False) &
            (self.df['_is_detail_header'] == False) &
            (self.df[self.symbol_col].astype(str).str.len() < self.MAX_SYMBOL_LENGTH)
        ].copy()

        # Yardımcı sütunları kaldır
        detail_df = self._remove_helper_columns(detail_df)

        # Boş satırları ve sütunları temizle
        detail_df = detail_df.dropna(how='all').dropna(axis=1, how='all')

        # Başlık satırlarını temizle
        if len(detail_df.columns) > 1:
            second_col = detail_df.columns[1]
            detail_df = detail_df[
                ~detail_df[second_col].astype(str).str.strip().isin(self.DETAIL_HEADERS)
            ]

        # Kolon isimlerini ata
        if len(detail_df.columns) == len(self.DETAIL_COLUMNS):
            detail_df.columns = self.DETAIL_COLUMNS
        else:
            print(f"Uyarı: Beklenen {len(self.DETAIL_COLUMNS)} sütun, "
                  f"mevcut {len(detail_df.columns)} sütun")

        return detail_df.reset_index(drop=True)

    def _remove_helper_columns(self, df):
        """Yardımcı sütunları kaldır"""
        helper_cols = ['_is_symbol_row', '_is_detail_header']
        return df.drop(columns=[col for col in helper_cols if col in df.columns])

    def _convert_to_old_format_summary(self, df):
        """Özet satırlarını eski formata dönüştür (8→12 sütun)"""
        if len(df.columns) != 8:
            print(f"Uyarı: Beklenmeyen sütun sayısı ({len(df.columns)})")
            return df

        cm = self.COLUMN_MAPPING
        new_df = pd.DataFrame({
            'Sembol': df.iloc[:, cm['symbol']],
            'Alanlar': df.iloc[:, cm['symbol']],
            'Alici_Net_Hacim': df.iloc[:, cm['alis_hacim']],
            'Alici_Ortalama': df.iloc[:, cm['alis_ort']],
            'Alici_Yuzdesi': df.iloc[:, cm['alis_yuzde']],
            'Alici_Net_Lot': pd.NA,  # Sonra hesaplanacak
            'Satanlar': df.iloc[:, cm['symbol']],
            'Satici_Net_Hacim': df.iloc[:, cm['satis_hacim']],
            'Satici_Ortalama': df.iloc[:, cm['satis_ort']],
            'Satici_Yuzdesi': df.iloc[:, cm['satis_yuzde']],
            'Satici_Net_Lot': pd.NA,  # Sonra hesaplanacak
            'Para_Giris_Cikis': df.iloc[:, cm['para_gc']]
        })

        return new_df.reset_index(drop=True)

    @classmethod
    def merge_summary_and_details(cls, summary_df, detail_df):
        """Özet ve detay verilerini birleştir"""
        all_cols = list(dict.fromkeys(
            list(summary_df.columns) + list(detail_df.columns)
        ))
        blocks = []

        for symbol in summary_df["Sembol"].unique():
            # Özet satırı
            summary_block = summary_df[summary_df["Sembol"] == symbol].copy()
            summary_block = summary_block.reindex(columns=all_cols)

            # Detay satırları
            detail_block = detail_df[detail_df["Sembol"] == symbol].copy()
            detail_block = detail_block.reindex(columns=all_cols)
            detail_block["Sembol"] = symbol

            # Satıcı değerlerini negatif yap
            cls._apply_seller_negatives(detail_block)

            # "Diğer" normalizasyonu
            cls._normalize_diger_labels(summary_block, detail_block)

            # "Diğer" hariç toplamları hesapla
            cls._calculate_totals(summary_block, detail_block)

            blocks.extend([summary_block, detail_block])


        result = pd.concat(blocks, ignore_index=True)
        # 🔥 En kritik satır → numeric dönüşüm,takas tarafında sorun oluşturunca bu class ı ekledim.
        result = cls.convert_numeric(result)
        return result

    @staticmethod
    def _apply_seller_negatives(detail_df):
        """Satıcı değerlerini negatif yap"""
        seller_cols = ["Satici_Net_Hacim", "Satici_Yuzdesi"]
        for col in seller_cols:
            if col in detail_df.columns:
                detail_df[col] = -detail_df[col].abs()

        # DİĞER satırlarında Satici_Net_Lot'u negatif yap
        if "Satici_Net_Lot" in detail_df.columns:
            mask_diger = detail_df["Alanlar"].apply(
                lambda x: PGCProcessor.normalize_text(x) == "DIGER"
            )
            detail_df.loc[mask_diger, "Satici_Net_Lot"] = (
                -detail_df.loc[mask_diger, "Satici_Net_Lot"].abs()
            )

    @staticmethod
    def _normalize_diger_labels(summary_df, detail_df):
        """Alanlar ve Satanlar sütunlarında 'Diğer' varyantlarını normalize et"""
        for col in ["Alanlar", "Satanlar"]:
            for df in [summary_df, detail_df]:
                if col in df.columns:
                    mask = df[col].apply(
                        lambda x: PGCProcessor.normalize_text(x) == "DIGER"
                    )
                    df.loc[mask, col] = "DIGER"

    @staticmethod
    def _calculate_totals(summary_df, detail_df):
        """DİĞER hariç toplamları hesapla"""
        # Normalize edilmiş kontrol kolonu
        detail_df["_temp_norm"] = detail_df["Alanlar"].apply(
            PGCProcessor.normalize_text
        )

        # DİĞER hariç detaylar
        detail_no_diger = detail_df[detail_df["_temp_norm"] != "DIGER"].copy()

        # Toplamları hesapla
        sum_cols = [
            "Alici_Net_Lot", "Satici_Net_Lot",
            "Alici_Net_Hacim", "Satici_Net_Hacim",
            "Alici_Yuzdesi", "Satici_Yuzdesi"
        ]

        for col in sum_cols:
            if col in detail_no_diger.columns:
                summary_df.loc[summary_df.index[0], col] = detail_no_diger[col].sum()

        # Geçici kolonu kaldır
        detail_df.drop(columns=["_temp_norm"], inplace=True)


    @staticmethod
    def convert_numeric(df):
        """Sayısal kolonları otomatik numeric'e çevirir"""
        numeric_cols = [
            'Alici_Net_Hacim', 'Alici_Ortalama', 'Alici_Yuzdesi', 'Alici_Net_Lot',
            'Satici_Net_Hacim', 'Satici_Ortalama', 'Satici_Yuzdesi', 'Satici_Net_Lot',
            'Para_Giris_Cikis'
        ]

        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        return df


# Kullanım örneği
def process_pgc_file(file_path):
    """PGC dosyasını işle ve birleştirilmiş sonucu döndür"""
    # Dosyayı oku
    df = pd.read_excel(file_path)

    # İşle
    processor = PGCProcessor(df)
    summary_df = processor.get_summary_rows()
    detail_df = processor.get_detail_rows()

    # Birleştir
    result_df = PGCProcessor.merge_summary_and_details(summary_df, detail_df)

    return result_df

# Yeni pgc_ddmm.xls hazırlanması
Günlük pgc(prime dan 20 kurum ve yalnızca hisseler seçilecek ve dosya adı pgc_ddmm.xlsx) nin bulunması ve Tarih alanının eklenmesi.

In [ ]:
# En yeni tarihli günlük pgc dosyasını bul
pgc_files = glob("pgc_*.xlsx")
if not pgc_files:
    raise FileNotFoundError("Hiç pgc_*.xlsx dosyası bulunamadı!")

pgc_latest_file = max(pgc_files, key=lambda x: int(x.split('_')[1].split('.')[0]))
print(f"📂 En yeni dosya: {pgc_latest_file}")

# Dosyayı işle ve tarih ekle
pgc = process_pgc_file(pgc_latest_file)  #process_pgc_file Fonksiyonunu çalıştır.

tarih_parcasi = os.path.basename(pgc_latest_file).split('_')[1].split('.')[0]
#yeni yıla geçince 31 inin kayıtlarında sıkıntı oldu
# tarih = datetime.strptime(f"{tarih_parcasi[:2]}-{tarih_parcasi[2:]}-{datetime.now().year}", '%d-%m-%Y')

gun = int(tarih_parcasi[:2])
ay = int(tarih_parcasi[2:])
yil = datetime.now().year

# bugun_yil = datetime.now().year

# # 🔐 YIL KURALI
# if gun == 31 and ay == 12:
#     yil = bugun_yil - 1
# else:
#     yil = bugun_yil

# tarih = datetime(year=yil, month=ay, day=gun)

if gun == 31 and ay == 12:
    yil -= 1

tarih = datetime.strptime(f"{gun:02d}-{ay:02d}-{yil}", "%d-%m-%Y")

pgc['Tarih'] = tarih

print(f"✅ Tarih eklendi: {tarih.strftime('%d-%m-%Y')}")

# Fonksiyonlar
Güncel dosyalar ve hesaplamalar için tanımlı fonksiyonlar

In [ ]:
def en_guncel_hisse_master_dosyasi(dizin="/content"):
    excel_files = glob(os.path.join(dizin, "HisseKurumSeriDurumlar_*.xlsx"))

    if not excel_files:
        return "/content/HisseKurumSeriDurumlar_01012000.xlsx"

    tarihli = {}
    for f in excel_files:
        match = re.search(r'HisseKurumSeriDurumlar_(\d{8})\.xlsx', os.path.basename(f))
        if match:
            tarihli[match.group(1)] = f

    if not tarihli:
        return "/content/HisseKurumSeriDurumlar_01012000.xlsx"

    return tarihli[max(tarihli.keys())]


def fiili_dolasim_yukle(excel_path, sheet_name='FiiliDolasim'):
    try:
        df = pd.read_excel(excel_path, sheet_name=sheet_name)

        if 'Borsa Kodu' in df.columns and 'Fiili Dolaşımdaki Pay Adedi' in df.columns:
            return df.set_index('Borsa Kodu')['Fiili Dolaşımdaki Pay Adedi'].astype(float).to_dict()

        print("⚠ FiiliDolasim sheet'inde gerekli sütunlar yok.")
        return {}

    except Exception as e:
        print(f"⚠ FiiliDolasim okuma hatası: {e}")
        return {}

#webden direkt almal için aşağıdaki fonksiyonu çalıştır.
def fiili_dolasim_yukle_from_df(FiiliDolasim_df):
    """
    FiiliDolasim_df DataFrame'inden sözlük oluşturur.
    { "AKBNK": 2790230471.02, ... }
    """
    try:
        if ("Borsa Kodu" in FiiliDolasim_df.columns and
            "Fiili Dolaşımdaki Pay Adedi" in FiiliDolasim_df.columns):

            df = FiiliDolasim_df.copy()

            # Borsa Kodu index olsun
            df = df.set_index("Borsa Kodu")

            # Sözlük oluştur
            return df["Fiili Dolaşımdaki Pay Adedi"].astype(float).to_dict()

        print("⚠ FiiliDolasim_df gerekli sütunlara sahip değil.")
        return {}

    except Exception as e:
        print(f"⚠ FiiliDolasim_df işleme hatası: {e}")
        return {}


def hisse_master_yukle(dizin="/content"):
    """
    En güncel HisseKurumSeriDurumlar dosyasını bulur ve
    FiiliDolasim sözlüğünü yükleyip ikisini birden döndürür.
    """
    path = en_guncel_hisse_master_dosyasi(dizin)

    # Dosya yoksa oluştur
    if not os.path.exists(path):
        Workbook().save(path)

    fiili_dict = fiili_dolasim_yukle(path)
    FiiliDolasim_dict=fiili_dolasim_yukle_from_df(FiiliDolasim_df) #webden dinamik olarak aldım

    print(f"📂 Kullanılan dosya: {path}")
    print(f"📊 FiiliDolasim yüklenen hisse sayısı: {len(fiili_dict)}")

    return path, fiili_dict

def dolasima_orani_hesapla(hisse_kodu, ardisik_toplam):
    fiili_dolasim = fiili_dolasim_dict.get(hisse_kodu, 0)
    if fiili_dolasim > 0:
        return round((ardisik_toplam / fiili_dolasim) * 100, 2)
    return 0.00


#Farklı Tarih formatları nedeniyle tarih kolonlarını bulmak için yapıldı.
def tarih_kolonlarini_bul(row_or_df_columns):
    """
    Farklı formatlardaki tarih kolonlarını otomatik yakalar.
    string veya pd.Timestamp olabilir.
    Geçerli formatlar:
        - DD-MM-YYYY
        - DD/MM/YYYY
        - YYYY-MM-DD
        - YYYY/MM/DD
    """

    patterns = [
        r'^\d{2}-\d{2}-\d{4}$',
        r'^\d{2}/\d{2}/\d{4}$',
        r'^\d{4}-\d{2}-\d{2}$',
        r'^\d{4}/\d{2}/\d{2}$',
    ]

    cols = []
    for c in row_or_df_columns:
        if isinstance(c, pd.Timestamp):
            cols.append(c)
        else:
            sc = str(c).strip()
            if any(re.match(p, sc) for p in patterns):
                cols.append(c)

    return cols

def safe_parse_date(col):
    """Tarihi güvenli bir şekilde parse eder"""
    from datetime import datetime

    if isinstance(col, pd.Timestamp):
        return col

    if pd.isna(col):
        return None

    try:
        col_str = str(col).strip()

        # Farklı formatları dene
        for fmt in ["%d-%m-%Y", "%d/%m-%Y", "%Y-%m-%d", "%Y/%m/%d"]:
            try:
                return datetime.strptime(col_str, fmt)
            except:
                continue

        # pd.to_datetime ile dene
        parsed = pd.to_datetime(col_str, dayfirst=True, errors='coerce')
        if not pd.isna(parsed):
            return parsed
    except:
        pass

    return None



def ardisik_getiri_hesapla(row, tum_veriler, debug=False):
    """
    Pivot satırındaki ArdışıkGetiri'yi hesaplar.
    Tarih kolonları farklı formatlarda olabilir veya pd.Timestamp olabilir.
    """
    sembol = row.name
    ard_gun = row.get('ArdışıkGün', np.nan)

    try:
        ard = int(ard_gun)
    except:
        if debug:
            print(f"⚠ {sembol}: ArdışıkGün yok veya geçersiz")
        return np.nan

    if ard == 0:
        if debug:
            print(f"⚠ {sembol}: ArdışıkGün 0")
        return np.nan

    # --- 1) Tarih kolonlarını bul ---
    tarih_kolonlari = tarih_kolonlarini_bul(row.index)
    if not tarih_kolonlari:
        if debug:
            print(f"⚠ {sembol}: Tarih kolonları bulunamadı")
        return np.nan

    # --- 2) Kronolojik sıraya göre sırala ---
    def kolon_to_date(c):
        if isinstance(c, pd.Timestamp):
            return c.date()
        try:
            return pd.to_datetime(c, dayfirst=True, errors='coerce').date()
        except:
            return pd.NaT

    tarih_kolonlari = sorted(
        [c for c in tarih_kolonlari if kolon_to_date(c) is not pd.NaT],
        key=lambda x: kolon_to_date(x)
    )

    son_idx = len(tarih_kolonlari) - 1
    bas_idx = max(0, son_idx - (abs(ard) - 1))

    son_col = tarih_kolonlari[son_idx]
    bas_col = tarih_kolonlari[bas_idx]

    son_dt = kolon_to_date(son_col)
    bas_dt = kolon_to_date(bas_col)

    # --- 3) tum_veriler ile eşleştir ---
    df_h = tum_veriler[tum_veriler['Hisse'] == sembol].copy()
    if df_h.empty:
        if debug:
            print(f"⚠ {sembol}: tum_veriler içinde bulunamadı")
        return np.nan

    if pd.api.types.is_datetime64_any_dtype(df_h['tarih']):
        df_h['tarih_date'] = df_h['tarih'].dt.date
    else:
        df_h['tarih_date'] = pd.to_datetime(df_h['tarih'], errors='coerce').dt.date

    df_son = df_h[df_h['tarih_date'] == son_dt]
    df_bas = df_h[df_h['tarih_date'] == bas_dt]

    if df_son.empty or df_bas.empty:
        if debug:
            print(f"⚠ {sembol}: Baş/son tarih verisi eksik")
            print(f"   Bas: {bas_dt} -> {len(df_bas)} satır")
            print(f"   Son: {son_dt} -> {len(df_son)} satır")
        return np.nan

    close_son = df_son['close'].iloc[0]
    close_bas = df_bas['close'].iloc[0]

    getir = (close_son / close_bas) - 1

    if debug:
        print(f"✓ {sembol}: {bas_dt} -> {son_dt}, ArdışıkGün:{ard}, Close:{close_bas}->{close_son}, Getiri:{getir*100:.2f}%")

    return round(getir * 100, 2)

def ardisik_gun_hesapla(row):
    """
    Satırdaki sadece tarih kolonlarını bularak ardışık gün hesaplar.
    """
    # Tarih kolonlarını bul
    tarih_kolonlari = tarih_kolonlarini_bul(row.index)

    if not tarih_kolonlari:
        return 0

    # Tarih kolonlarını parse et ve sırala
    tarih_tuples = []
    for col in tarih_kolonlari:
        parsed = safe_parse_date(col)
        if parsed is not None:
            tarih_tuples.append((col, parsed))

    if not tarih_tuples:
        return 0

    # Parse edilmiş tarihe göre sırala
    tarih_tuples.sort(key=lambda x: x[1])
    tarih_kolonlari = [col for col, _ in tarih_tuples]

    # Değerleri al
    values = [0 if pd.isna(row[c]) else row[c] for c in tarih_kolonlari]

    # Ardışık gün hesabı
    ard_pos = ard_neg = 0
    for v in reversed(values):
        if v == 0:
            break
        if v > 0:
            if ard_neg:
                break
            ard_pos += 1
        else:
            if ard_pos:
                break
            ard_neg += 1

    return ard_pos if ard_pos else -ard_neg if ard_neg else 0


def ardisik_toplam_hesapla(row, debug=False):
    """
    ArdışıkToplam hesaplar.
    """
    from datetime import datetime

    # ArdışıkGün
    ard_gun = row.get('ArdışıkGün', 0)
    try:
        gun_sayisi = int(ard_gun)
    except (TypeError, ValueError):
        gun_sayisi = 0

    if gun_sayisi == 0:
        return 0

    pozitif_mi = gun_sayisi > 0
    gun_sayisi = abs(gun_sayisi)

    # Tarih kolonlarını bul
    date_columns = tarih_kolonlarini_bul(row.index)
    if not date_columns:
        if debug:
            print(f"⚠ {row.name}: Tarih kolonları bulunamadı")
        return 0

    # Parse et ve sırala
    tarih_tuples = []
    for col in date_columns:
        parsed = safe_parse_date(col)
        if parsed is not None:
            tarih_tuples.append((col, parsed))

    if not tarih_tuples:
        return 0

    tarih_tuples.sort(key=lambda x: x[1])
    date_columns = [col for col, _ in tarih_tuples]

    # Son N günü al
    date_columns_son = date_columns[-gun_sayisi:]

    # Değerleri oku
    values = [row[c] if pd.notna(row[c]) else 0 for c in date_columns_son]

    # Pozitif/Negatif filtreleme
    if pozitif_mi:
        values = [v for v in values if v > 0]
    else:
        values = [v for v in values if v < 0]

    return sum(values)


In [ ]:
#Güncel dosya-hisse master ve fiili dolaşımı yükle
# hissemaster_path, fiili_dolasim_dict = hisse_master_yukle()
#Fiili dolaşım webdebi verilerini yükle
fiili_dolasim_dict=fiili_dolasim_yukle_from_df(FiiliDolasim_df)

In [ ]:
def en_guncel_parquet_dosyasini_bul(dosya_dizini=".", dosya_bulunamadi_mesaji=True):
    """
    En güncel PgcGunlukArsiv parquet dosyasını bulur ve okur.
    Args:
        dosya_dizini (str): Dosyaların aranacağı dizin
        dosya_bulunamadi_mesaji (bool): Uyarı mesajı gösterilsin mi?
    Returns:
        tuple: (DataFrame, dosya_bilgileri_dict) veya (boş DataFrame, None)

    Raises:
        FileNotFoundError: Dosya okunamazsa
    """
    try:
        # Dosya arama
        arama_pattern = os.path.join(dosya_dizini, "PgcGunlukArsiv_*.parquet")
        parquet_files = glob(arama_pattern)

        if not parquet_files:
            if dosya_bulunamadi_mesaji:
                print("⚠️ Mevcut parquet dosyası bulunamadı, yeni oluşturulacak...")
            return pd.DataFrame(), None

        # Tarih çıkarma fonksiyonu
        def tarih_cikar(dosya_adi):
            match = re.search(r'PgcGunlukArsiv_(\d{8})\.parquet', dosya_adi)
            return match.group(1) if match else "00000000"

        # En güncel dosyayı bul
        en_guncel_dosya = max(parquet_files, key=tarih_cikar)
        dosya_dizini = os.path.dirname(os.path.abspath(en_guncel_dosya))
        dosya_tarihi = tarih_cikar(en_guncel_dosya)

        print(f"📂 Son arşiv dosyası: {en_guncel_dosya}")

        # Veriyi oku
        veri_df = pd.read_parquet(en_guncel_dosya)
        print(f"📊 Mevcut arşiv boyutu: {veri_df.shape}")

        # Dosya bilgilerini hazırla
        dosya_bilgileri = {
            'dosya_adi': en_guncel_dosya,
            'dizin': dosya_dizini,
            'tarih': dosya_tarihi,
            'dosya_sayisi': len(parquet_files)
        }

        return veri_df, dosya_bilgileri

    except Exception as e:
        print(f"❌ Dosya okuma hatası: {e}")
        return pd.DataFrame(), None

# Kullanımı:
# PgcGunlukArsiv_df, dosya_bilgileri = en_guncel_parquet_dosyasini_bul()

# if dosya_bilgileri:
#     print(f"Bulunan dosya: {dosya_bilgileri['dosya_adi']}")
#     print(f"Dosya tarihi: {dosya_bilgileri['tarih']}")
#     print(f"Dizin Adı: {dosya_bilgileri['dizin']}")

# **Günlük pgc nin "PgcGunlukArsiv_*.parquet" eklenmesi/düzenlenmesi**



In [ ]:
# === 1. En güncel parquet dosyasını bul ===
PgcGunlukArsiv_df, parquet_info = en_guncel_parquet_dosyasini_bul()

if parquet_info:
    latest_file = parquet_info["dosya_adi"]
    base_dir = parquet_info["dizin"]
else:
    latest_file = None
    base_dir = "."

# === 2. Yeni pgc'yi işle ===
yeni_pgc_df = pgc.copy()

# === 3. Kolon tiplerini arşiv referansına göre dönüştür (sadece arşiv boş değilse) ===
if not PgcGunlukArsiv_df.empty:
    for col in PgcGunlukArsiv_df.columns:
        if col in yeni_pgc_df.columns:
            eski_tur = PgcGunlukArsiv_df[col].dtype
            try:
                if pd.api.types.is_integer_dtype(eski_tur):
                    yeni_pgc_df[col] = pd.to_numeric(yeni_pgc_df[col], errors="coerce").astype("Int64")
                elif pd.api.types.is_float_dtype(eski_tur):
                    yeni_pgc_df[col] = pd.to_numeric(yeni_pgc_df[col], errors="coerce").astype("Float64")
                elif pd.api.types.is_datetime64_any_dtype(eski_tur):
                    yeni_pgc_df[col] = pd.to_datetime(yeni_pgc_df[col], format='%d-%m-%y', errors="coerce")
                elif pd.api.types.is_bool_dtype(eski_tur):
                    yeni_pgc_df[col] = yeni_pgc_df[col].astype("boolean")
                else:
                    yeni_pgc_df[col] = yeni_pgc_df[col].astype("string")
            except Exception as e:
                print(f"⚠️ {col} dönüşürken hata: {e}")

    # Eksik sütunları ekle
    for col in PgcGunlukArsiv_df.columns:
        if col not in yeni_pgc_df.columns:
            yeni_pgc_df[col] = pd.NA

# === 4. Yeni veri + eski veriyi birleştir ===
if not PgcGunlukArsiv_df.empty:
    birlesik_df = pd.concat([PgcGunlukArsiv_df, yeni_pgc_df], ignore_index=True)
    print(f"📥 Birleştirme sonrası: {birlesik_df.shape}")
else:
    birlesik_df = yeni_pgc_df.copy()
    print(f"📥 İlk veri yüklendi: {birlesik_df.shape}")

# === 5. Duplicate temizliği ===
before = len(birlesik_df)
birlesik_df = birlesik_df.drop_duplicates(keep="first")
after = len(birlesik_df)
print(f"🧹 Duplicate kayıtlar temizlendi. {before - after} satır silindi. Yeni boyut: {after}")

# === 6. Tarih sütunu kesin datetime olsun ===
if "Tarih" not in birlesik_df.columns:
    raise ValueError("Tarih sütunu bulunamadı!")

birlesik_df["Tarih"] = pd.to_datetime(birlesik_df["Tarih"], errors="coerce")

# === 7. 30 Gün sınırı ===
benzersiz_tarihler = birlesik_df["Tarih"].dt.normalize().unique()
print(f"📅 Mevcut benzersiz tarih sayısı: {len(benzersiz_tarihler)}")

if len(benzersiz_tarihler) > 30:
    tarih_sayilari = birlesik_df["Tarih"].dt.normalize().value_counts().sort_index()
    silinecek_tarihler = tarih_sayilari.head(len(benzersiz_tarihler) - 30).index

    birlesik_df = birlesik_df[~birlesik_df["Tarih"].dt.normalize().isin(silinecek_tarihler)]

    print(f"✂️ {len(silinecek_tarihler)} eski tarih silindi. Yeni boyut: {birlesik_df.shape}")
else:
    print(f"✅ 30 gün sınırı aşılmamış.")

# === 8. Yeni dosya ismi (max tarih) ===
max_date = birlesik_df["Tarih"].max()
if pd.isna(max_date):
    raise ValueError("Geçerli bir Tarih bulunamadı!")

date_str = max_date.strftime("%Y%m%d")
yeni_parquetDosya = f"PgcGunlukArsiv_{date_str}.parquet"

print(f"📅 En yeni tarih: {max_date.strftime('%d-%m-%Y')}")
print(f"💾 Kaydedilecek dosya: {yeni_parquetDosya}")

# === 9. Kaydet ===
birlesik_df.to_parquet(yeni_parquetDosya, index=False)
print(f"✅ Güncel birikimli parquet kaydedildi: {yeni_parquetDosya}")

# === 10. Günlük özet ===
ozet_df = (
    birlesik_df
    .groupby(birlesik_df["Tarih"].dt.normalize())
    .size()
    .reset_index(name="KayıtSayısı")
    .rename(columns={"Tarih": "Tarih_datetime"})
)

ozet_df = ozet_df.sort_values("Tarih_datetime")
ozet_df["Tarih"] = ozet_df["Tarih_datetime"].dt.strftime("%d-%m-%Y")

print("\n📈 Günlük kayıt özeti:")
print(ozet_df[["Tarih", "KayıtSayısı"]].to_string(index=False))

# === 11. Eski dosyaları sil ===
if latest_file and os.path.exists(latest_file):
    os.remove(latest_file)
    os.remove(pgc_latest_file)
    print(f"🗑️ Eski parquet silindi: {latest_file}")
    print(f"🗑️ Eski pgc silindi: {pgc_latest_file}")


In [ ]:
PgcGunlukArsiv_df, parquet_info = en_guncel_parquet_dosyasini_bul()

In [ ]:
# ozet_df = (
#     PgcGunlukArsiv_df
#     .groupby(PgcGunlukArsiv_df["Tarih"].dt.normalize())
#     .size()
#     .reset_index(name="KayıtSayısı")
#     .rename(columns={"Tarih": "Tarih_datetime"})
# )

# ozet_df = ozet_df.sort_values("Tarih_datetime")
# ozet_df["Tarih"] = ozet_df["Tarih_datetime"].dt.strftime("%d-%m-%Y")

# print("\n📈 Günlük kayıt özeti:")
# print(ozet_df[["Tarih", "KayıtSayısı"]].to_string(index=False))

In [ ]:
# PgcGunlukArsiv_df, parquet_info = en_guncel_parquet_dosyasini_bul()

# Güncel PgcGunlukArsiv parquet dosyasından Hisse-Tarih bazlı genel 'Para_Giris_Cikis' Pivot tablosu oluşur

In [ ]:
#boş ve sıfır hatası için aşağıdaki denenebilir.

# Parquet okurken temizlik
PgcGunlukArsiv_df, parquet_info = en_guncel_parquet_dosyasini_bul()

# Tüm numeric kolonları temizle
numeric_cols = [
    'Alici_Net_Hacim', 'Alici_Ortalama', 'Alici_Yuzdesi', 'Alici_Net_Lot',
    'Satici_Net_Hacim', 'Satici_Ortalama', 'Satici_Yuzdesi', 'Satici_Net_Lot',
    'Para_Giris_Cikis'
]

for col in numeric_cols:
    if col in PgcGunlukArsiv_df.columns:
        PgcGunlukArsiv_df[col] = PgcGunlukArsiv_df[col].replace('', pd.NA)
        PgcGunlukArsiv_df[col] = pd.to_numeric(PgcGunlukArsiv_df[col], errors='coerce')

print("✓ Parquet temizlendi")

# Pivot oluşturmak
Ozet Pgc ve NetLot için

In [ ]:
#ilk 5 kuruma göre yapıyoruz,yoksa 20 ye göre yaparsak tümü olduğu için pgc,lot anlamlı olmaz ,piyasa genel olarak ilk5 e bakıyor.
def pgc_hisse_detayli_ozet_olustur(PgcGunlukArsiv_df, top_n=5):
    """
    Hisse bazında Top N ve geri kalan (DIGER dahil) kurumların detaylı özetini oluşturur.

    VERİ YAPISI:
    - Alici_Net_Hacim: Pozitif
    - Satici_Net_Hacim: Negatif
    - Alici_Yuzdesi: Pozitif
    - Satici_Yuzdesi: Negatif
    - Alici_Net_Lot: Pozitif
    - Satici_Net_Lot: Kurumlar için pozitif, DIGER için negatif

    Parameters:
    -----------
    PgcGunlukArsiv_df : DataFrame
        Günlük arşiv verisi
    top_n : int
        Kaç büyük kurum ayrı hesaplanacak (varsayılan: 5)

    Returns:
    --------
    DataFrame: Detaylı özet (Top N + DIGER_TOPLAM)
    """

    # 1. Genel bilgi satırlarını hariç tut
    filtre = ~(
        (PgcGunlukArsiv_df['Sembol'] == PgcGunlukArsiv_df['Alanlar']) &
        (PgcGunlukArsiv_df['Sembol'] == PgcGunlukArsiv_df['Satanlar'])
    )

    df_temiz = PgcGunlukArsiv_df[filtre].copy()

    # 2. Numeric kolonları temizle
    numeric_cols = [
        'Alici_Net_Hacim', 'Alici_Yuzdesi', 'Alici_Net_Lot',
        'Satici_Net_Hacim', 'Satici_Yuzdesi', 'Satici_Net_Lot'
    ]

    for col in numeric_cols:
        df_temiz[col] = df_temiz[col].replace('', pd.NA)
        df_temiz[col] = pd.to_numeric(df_temiz[col], errors='coerce').fillna(0)

    # 3. Her hisse-tarih için detaylı hesaplama
    ozet_liste = []

    for (sembol, tarih), group in df_temiz.groupby(['Sembol', 'Tarih']):
        # Sembol uzunluğu kontrolü
        if len(str(sembol)) >= 6:
            continue

        # =====================================================================
        # ALICI TARAFI
        # =====================================================================
        # Top N kurum (DIGER dahil değil, en büyük hacme göre)
        alici_diger_haric = group[group['Alanlar'] != 'DIGER']
        alici_top_n = alici_diger_haric.nlargest(top_n, 'Alici_Net_Hacim')

        # Top N dışında kalanlar + DIGER (16 kurum + DIGER)
        alici_top_n_kurumlar = set(alici_top_n['Alanlar'].tolist())
        alici_geri_kalan = group[~group['Alanlar'].isin(alici_top_n_kurumlar)]

        # Top N toplamları
        alici_top = {
            'Alici_Net_Hacim_TopN': alici_top_n['Alici_Net_Hacim'].sum(),
            'Alici_Yuzdesi_TopN': alici_top_n['Alici_Yuzdesi'].sum(),
            'Alici_Net_Lot_TopN': alici_top_n['Alici_Net_Lot'].sum()
        }

        # Geri kalan + DIGER toplamları
        alici_diger = {
            'Alici_Net_Hacim_Diger': alici_geri_kalan['Alici_Net_Hacim'].sum(),
            'Alici_Yuzdesi_Diger': alici_geri_kalan['Alici_Yuzdesi'].sum(),
            'Alici_Net_Lot_Diger': alici_geri_kalan['Alici_Net_Lot'].sum()
        }

        # =====================================================================
        # SATICI TARAFI
        # =====================================================================
        # Top N kurum (DIGER dahil değil, mutlak değere göre en büyükler)
        satici_diger_haric = group[group['Satanlar'] != 'DIGER'].copy()
        satici_diger_haric['Satici_Net_Hacim_Abs'] = satici_diger_haric['Satici_Net_Hacim'].abs()
        satici_top_n = satici_diger_haric.nlargest(top_n, 'Satici_Net_Hacim_Abs')

        # Top N dışında kalanlar + DIGER (16 kurum + DIGER)
        satici_top_n_kurumlar = set(satici_top_n['Satanlar'].tolist())
        satici_geri_kalan = group[~group['Satanlar'].isin(satici_top_n_kurumlar)]

        # Top N toplamları
        # Hacim ve Yüzde: Zaten negatif, direkt topla
        # Lot: Kurumlar pozitif ama DIGER negatif, önce negatif yap
        satici_top = {
            'Satici_Net_Hacim_TopN': satici_top_n['Satici_Net_Hacim'].sum(),
            'Satici_Yuzdesi_TopN': satici_top_n['Satici_Yuzdesi'].sum(),
            'Satici_Net_Lot_TopN': -satici_top_n['Satici_Net_Lot'].abs().sum()  # Negatif yap
        }

        # Geri kalan + DIGER toplamları
        # Hacim ve Yüzde: Zaten negatif, direkt topla
        # Lot: Kurumlar pozitif ama DIGER negatif, toplam doğru olsun diye önce negatif yap
        satici_diger_lot_toplam = 0
        for _, row in satici_geri_kalan.iterrows():
            if row['Satanlar'] == 'DIGER':
                # DIGER zaten negatif
                satici_diger_lot_toplam += row['Satici_Net_Lot']
            else:
                # Diğer kurumlar pozitif, negatif yap
                satici_diger_lot_toplam += -abs(row['Satici_Net_Lot'])

        satici_diger = {
            'Satici_Net_Hacim_Diger': satici_geri_kalan['Satici_Net_Hacim'].sum(),
            'Satici_Yuzdesi_Diger': satici_geri_kalan['Satici_Yuzdesi'].sum(),
            'Satici_Net_Lot_Diger': satici_diger_lot_toplam
        }

        # =====================================================================
        # TÜM KURUMLARIN TOPLAMI
        # =====================================================================
        # Tüm kurumlar için lot hesabı
        satici_tum_lot_toplam = 0
        for _, row in group.iterrows():
            if row['Satanlar'] == 'DIGER':
                satici_tum_lot_toplam += row['Satici_Net_Lot']
            else:
                satici_tum_lot_toplam += -abs(row['Satici_Net_Lot'])

        alici_tum = {
            'Alici_Net_Hacim_Tum': group['Alici_Net_Hacim'].sum(),
            'Alici_Yuzdesi_Tum': group['Alici_Yuzdesi'].sum(),
            'Alici_Net_Lot_Tum': group['Alici_Net_Lot'].sum()
        }

        satici_tum = {
            'Satici_Net_Hacim_Tum': group['Satici_Net_Hacim'].sum(),
            'Satici_Yuzdesi_Tum': group['Satici_Yuzdesi'].sum(),
            'Satici_Net_Lot_Tum': satici_tum_lot_toplam
        }

        # =====================================================================
        # FARKLAR (NET AKIŞ)
        # =====================================================================
        # Satıcı değerleri artık hepsi negatif, direkt toplama yapıyoruz
        farklar = {
            'Para_Giris_Cikis_TopN': alici_top['Alici_Net_Hacim_TopN'] + satici_top['Satici_Net_Hacim_TopN'],
            'Para_Giris_Cikis_Diger': alici_diger['Alici_Net_Hacim_Diger'] + satici_diger['Satici_Net_Hacim_Diger'],
            'Para_Giris_Cikis_Tum': alici_tum['Alici_Net_Hacim_Tum'] + satici_tum['Satici_Net_Hacim_Tum'],

            'Net_Lot_TopN': alici_top['Alici_Net_Lot_TopN'] + satici_top['Satici_Net_Lot_TopN'],
            'Net_Lot_Diger': alici_diger['Alici_Net_Lot_Diger'] + satici_diger['Satici_Net_Lot_Diger'],
            'Net_Lot_Tum': alici_tum['Alici_Net_Lot_Tum'] + satici_tum['Satici_Net_Lot_Tum']
        }

        # Tüm verileri birleştir
        kayit = {
            'Sembol': sembol,
            'Tarih': tarih,
            **alici_top,
            **alici_diger,
            **alici_tum,
            **satici_top,
            **satici_diger,
            **satici_tum,
            **farklar
        }

        ozet_liste.append(kayit)

    ozet_df = pd.DataFrame(ozet_liste)

    return ozet_df


def pivot_ve_analiz_coklu(ozet_df, pivot_tanimlari, debug=False):
    """
    Birden fazla pivot tablo oluşturur ve ardışık analizleri ekler.

    Parameters:
    -----------
    ozet_df : DataFrame
        Detaylı özet verisi
    pivot_tanimlari : dict
        {sheet_adi: deger_kolonu} formatında pivot tanımları

    Returns:
    --------
    dict: {sheet_adi: pivot_df} formatında pivot tabloları
    """
    pivotlar = {}

    for sheet_adi, deger_kolonu in pivot_tanimlari.items():
        # Pivot oluştur
        pivot_df = ozet_df.pivot_table(
            index='Sembol',
            columns='Tarih',
            values=deger_kolonu,
            aggfunc='first'
        )

        # Tarihleri sırala
        pivot_df = pivot_df.sort_index(axis=1, ascending=True)

        # Tarih kolonlarını formatla
        pivot_df.columns = pivot_df.columns.strftime('%d-%m-%Y')

        # 0 değerlerini NaN yap
        pivot_df = pivot_df.replace(0, np.nan)

        # Ardışık analizleri ekle
        pivot_df['ArdışıkGün'] = pivot_df.apply(ardisik_gun_hesapla, axis=1)
        pivot_df['ArdışıkToplam'] = pivot_df.apply(lambda row: ardisik_toplam_hesapla(row, debug=False),axis=1)
        pivot_df['ArdışıkGetiri'] = pivot_df.apply(lambda row: ardisik_getiri_hesapla(row, tum_veriler, debug=False),axis=1) #debug ardışık getiri için

        pivotlar[sheet_adi] = pivot_df

    return pivotlar

# =============================================================================
# KULLANIM
# =============================================================================

print("="*70)
print("DETAYLI ÖZET OLUŞTURULUYOR")
print("="*70)

top_n = 5
ozet_df = pgc_hisse_detayli_ozet_olustur(PgcGunlukArsiv_df, top_n=top_n)

print(f"✓ {len(ozet_df)} kayıt oluşturuldu")
print(f"✓ Kolonlar: {len(ozet_df.columns)} adet")
pivot_tanimlari = {
    f'Pgc({top_n})_Ozet': 'Para_Giris_Cikis_TopN',
    f'Netlot({top_n})_Ozet': 'Net_Lot_TopN',
}

# # 2. Pivot tanımları örnekleri
# pivot_tanimlari = {
#     'PGC_TopN': 'Para_Giris_Cikis_TopN',
#     'PGC_Diger': 'Para_Giris_Cikis_Diger',
#     'PGC_Tum': 'Para_Giris_Cikis_Tum',
#     'NetLot_TopN': 'Net_Lot_TopN',
#     'NetLot_Diger': 'Net_Lot_Diger',
#     'NetLot_Tum': 'Net_Lot_Tum',
# }


# 3. Tüm pivotları oluştur
print(f"\n{len(pivot_tanimlari)} adet pivot oluşturuluyor...")
pivotlar = pivot_ve_analiz_coklu(ozet_df, pivot_tanimlari, debug=False) #debug ardışık getiri için koydum.

for sheet_adi, pivot_df in pivotlar.items():
    print(f"✓ {sheet_adi}: {pivot_df.shape[0]} hisse x {pivot_df.shape[1]-2} tarih ")

# # 4. Excel'e kaydet,en sonda hepsinin güncellendiği yerde kaydediyorum.
# print("\nExcel'e kaydediliyor...")
# with pd.ExcelWriter(hissemaster_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     for sheet_adi, pivot_df in pivotlar.items():
#         pivot_df.fillna(0).to_excel(writer, sheet_name=sheet_adi)

# print("✓ Tüm pivotlar Excel'e kaydedildi")

# 5. Hızlı özet
print("\n" + "="*33)
print("En çok ardışık pgc toplamı")
print("="*33)
pgc_topn = pivotlar[f'Pgc({top_n})_Ozet']
print(pgc_topn.nlargest(10, 'ArdışıkToplam')[['ArdışıkGün', 'ArdışıkToplam']])

print("\n" + "="*33)
print("En çok ardışık pgc gün sayısı")
print("="*33)
print(pgc_topn.nlargest(10, 'ArdışıkGün')[['ArdışıkGün', 'ArdışıkToplam']])

print("\n" + "="*32)
print("En çok ardışık netLot gün sayısı")
print("="*32)
netLot_topn = pivotlar[f'Netlot({top_n})_Ozet']
print(netLot_topn.nlargest(10, 'ArdışıkGün')[['ArdışıkGün', 'ArdışıkToplam']])

In [ ]:
# # Tek hisse test etmek için

# def debug_hisse_getir_trading(sembol, pivot_df, tum_veriler):
#     """
#     Tek bir hisse için ArdışıkGetiri hesaplamasını adım adım gösterir.

#     Parameters:
#     ----------
#     sembol : str
#         Hisse kodu
#     pivot_df : DataFrame
#         Pivot DataFrame (index=Sembol)
#     tum_veriler : DataFrame
#         Tüm veriler, ['tarih','Hisse','close',...]

#     Returns:
#     -------
#     dict : detaylı bilgi
#     """
#     if sembol not in pivot_df.index:
#         print(f"Hata: {sembol} pivot_df içinde bulunamadı.")
#         return None

#     row = pivot_df.loc[sembol]
#     ard_gun = row.get('ArdışıkGün', np.nan)

#     if pd.isna(ard_gun) or int(ard_gun) == 0:
#         print(f"{sembol}: ArdışıkGün yok veya 0.")
#         return None

#     ard = abs(int(ard_gun))

#     # Pivot tarih kolonları
#     tarih_kolonlari = [c for c in row.index if re.match(r'\d{2}-\d{2}-\d{4}', str(c))]
#     tarih_kolonlari = sorted(tarih_kolonlari, key=lambda x: pd.to_datetime(x, format='%d-%m-%Y'))

#     son_idx = len(tarih_kolonlari) - 1
#     bas_idx = max(0, son_idx - (ard - 1))  # 🔹 trading-day mantığı

#     son_col = tarih_kolonlari[son_idx]
#     bas_col = tarih_kolonlari[bas_idx]

#     son_dt = pd.to_datetime(son_col, format='%d-%m-%Y').date()
#     bas_dt = pd.to_datetime(bas_col, format='%d-%m-%Y').date()

#     # Hissenin tüm verileri
#     df_h = tum_veriler[tum_veriler['Hisse'] == sembol].copy()
#     df_h['tarih_date'] = df_h['tarih'].dt.date

#     close_son = df_h.loc[df_h['tarih_date'] == son_dt, 'close'].iloc[0]
#     close_bas = df_h.loc[df_h['tarih_date'] == bas_dt, 'close'].iloc[0]

#     getir = (close_son / close_bas) - 1
#     getir_pct = getir * 100

#     print(f"\n🔹 Debug {sembol} 🔹")
#     print(f"Pivot son tarih: {son_col} -> close = {close_son}")
#     print(f"Pivot başlangıç tarih: {bas_col} -> close = {close_bas}")
#     print(f"ArdışıkGün: {ard}")
#     print(f"Getiri = (son/bas - 1) = {getir:.6f}  -> %{getir_pct:.3f}\n")

#     return {
#         'sembol': sembol,
#         'son_pivot_col': son_col,
#         'bas_pivot_col': bas_col,
#         'close_son': close_son,
#         'close_bas': close_bas,
#         'getiri': getir
#     }


# # örn pivotlar['Pgc_Ozet'] diye bir pivot df varsa:
# pivot_test_df = pivotlar[f'Pgc({top_n})_Ozet']

# # Tek hisse debug
# debug_hisse_getir_trading('SASA', pivot_test_df, tum_veriler)

In [ ]:
# PgcGunlukArsiv_df[(PgcGunlukArsiv_df['Sembol'] == 'SASA') & (PgcGunlukArsiv_df['Tarih'] == '2025-11-21')]

# **Kurum bazında ayrı ayrı matrisleri oluşturmak için.**

In [ ]:
def tarih_sutunu_ekle(pgc_df, tarih):
    """
    PGC DataFrame'ine tarih kolonu ekler.

    Parameters:
    -----------
    pgc_df : DataFrame
        İşlenmiş PGC verisi
    tarih : str veya datetime
        Tarih bilgisi

    Returns:
    --------
    DataFrame: Tarih kolonu eklenmiş PGC verisi
    """
    df = pgc_df.copy()

    # Tarih formatını standardize et
    if isinstance(tarih, str):
        tarih = pd.to_datetime(tarih)

    df['Tarih'] = tarih

    return df


def parquet_to_gunluk_pgc_list(parquet_df, tarih_kolonu='Tarih'):
    """
    Parquet dosyasından günlük PGC dataframe'lerini ayırır.

    Parameters:
    -----------
    parquet_df : DataFrame
        Parquet dosyasından okunan birleşik veri
    tarih_kolonu : str
        Tarih bilgisini içeren kolon adı

    Returns:
    --------
    dict: {tarih: pgc_dataframe} formatında sözlük
    """

    gunluk_veriler = {}

    # Benzersiz tarihleri al
    tarihler = sorted(parquet_df[tarih_kolonu].unique())

    for tarih in tarihler:
        # O tarihe ait verileri filtrele
        gunluk_df = parquet_df[parquet_df[tarih_kolonu] == tarih].copy()

        # Tarih kolonunu kaldır (işlenmiş PGC formatına dön)
        if tarih_kolonu in gunluk_df.columns:
            gunluk_df = gunluk_df.drop(columns=[tarih_kolonu])

        gunluk_veriler[tarih] = gunluk_df

    return gunluk_veriler


def kurum_bazli_zaman_serisi_matrisi(parquet_df, tarih_kolonu='Tarih'):
    """
    Her kurum için hisse x tarih matrisini oluşturur.

    Parameters:
    -----------
    parquet_df : DataFrame
        Parquet dosyasından okunan birleşik veri
    tarih_kolonu : str
        Tarih bilgisini içeren kolon adı

    Returns:
    --------
    dict: {kurum_adi: DataFrame(hisse x tarih)} formatında sözlük
    """

    # Veri temizliği: Boş string'leri NaN'e çevir
    df = parquet_df.copy()

    # Numeric kolonlardaki boş string'leri NaN yap
    numeric_cols = ['Alici_Net_Lot', 'Satici_Net_Lot']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].replace('', pd.NA)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Genel bilgi satırlarını hariç tut
    genel_filtre = ~((df['Sembol'] == df['Alanlar']) &
                     (df['Sembol'] == df['Satanlar']))

    pgc_filtreli = df[genel_filtre].copy()

    # Alıcı tarafı işlemleri
    alici_df = pgc_filtreli[['Sembol', 'Alanlar', 'Alici_Net_Lot', tarih_kolonu]].copy()
    alici_df.columns = ['Hisse', 'Kurum', 'Net (Adet)', 'Tarih']
    alici_df = alici_df[alici_df['Kurum'].notna()]

    # Satıcı tarafı işlemleri (negatif yap)
    satici_df = pgc_filtreli[['Sembol', 'Satanlar', 'Satici_Net_Lot', tarih_kolonu]].copy()
    satici_df.columns = ['Hisse', 'Kurum', 'Net (Adet)', 'Tarih']
    satici_df = satici_df[satici_df['Kurum'].notna()]
    satici_df['Net (Adet)'] = -satici_df['Net (Adet)'].abs()

    # Birleştir
    tum_islemler = pd.concat([alici_df, satici_df], ignore_index=True)

    # NaN'leri temizle
    tum_islemler = tum_islemler[tum_islemler['Net (Adet)'].notna()]

    # Aynı Hisse-Kurum-Tarih kombinasyonunu topla
    tum_islemler = tum_islemler.groupby(
        ['Hisse', 'Kurum', 'Tarih'],
        as_index=False
    )['Net (Adet)'].sum()

    # Sıfırları filtrele
    tum_islemler = tum_islemler[
        (tum_islemler['Net (Adet)'].notna()) &
        (tum_islemler['Net (Adet)'] != 0)
    ]

    # Benzersiz kurumları al
    kurumlar = sorted(tum_islemler['Kurum'].unique())

    # Her kurum için ayrı matris oluştur
    kurum_matrisleri = {}

    for kurum in kurumlar:
        # Bu kuruma ait verileri filtrele
        kurum_df = tum_islemler[tum_islemler['Kurum'] == kurum].copy()

        # Pivot: Hisse x Tarih
        matris = kurum_df.pivot(
            index='Hisse',
            columns='Tarih',
            values='Net (Adet)'
        )

        # Tarihleri sırala
        matris = matris[sorted(matris.columns)]

        # Hisseleri alfabetik sırala
        matris = matris.sort_index()

        kurum_matrisleri[kurum] = matris

    return kurum_matrisleri

def kurum_matrisi_analizli(matris_df, max_tarih_sayisi=30, fiili_dolasim_dict=None):
    """
    Kurum matrisine ardışık gün ve toplam kolonlarını ekler.
    """
    df = matris_df.copy()

    # Tarih kolonlarını al ve sırala
    tarih_kolonlari = list(df.columns)

    # Tarih kolonlarını parse et ve sırala
    tarih_tuples = []
    for col in tarih_kolonlari:
        parsed = safe_parse_date(col)
        if parsed is not None:
            tarih_tuples.append((col, parsed))

    if not tarih_tuples:
        print("⚠ Hiç geçerli tarih kolonu bulunamadı!")
        return df

    # Parse edilmiş tarihe göre sırala
    tarih_tuples.sort(key=lambda x: x[1])
    sirali_tarih_kolonlari = [col for col, _ in tarih_tuples]

    # DataFrame'i yeniden sırala
    df = df[sirali_tarih_kolonlari]

    # Sembol uzunluğu 6'dan küçük olanları filtrele
    df = df[df.index.str.len() < 6]

    # Ardışık Gün hesapla
    df['ArdışıkGün'] = df.apply(lambda row: ardisik_gun_hesapla(row), axis=1)

    # Ardışık Toplam hesapla
    df['ArdışıkToplam'] = df.apply(lambda row: ardisik_toplam_hesapla(row, debug=False), axis=1)

    # Dolaşıma Oranı hesapla
    if fiili_dolasim_dict:
        try:
            df['DolaşımaOranı'] = df.apply(
                lambda row: dolasima_orani_hesapla(row.name, row['ArdışıkToplam']),
                axis=1
            )
        except NameError:
            df['DolaşımaOranı'] = 0.00
    else:
        df['DolaşımaOranı'] = 0.00

    # Ardışık Getiri hesapla
    try:
        df['ArdışıkGetiri'] = df.apply(lambda row: ardisik_getiri_hesapla(row, tum_veriler, debug=False),axis=1)
    except (NameError, Exception):
        df['ArdışıkGetiri'] = np.nan

    return df



def kurum_ozet_matrisi_olustur_v2(analizli_kurum_tablolari, tip="AT"):
    """
    tip = 'AT' → ArdışıkToplam üzerinden özet
    tip = 'AG' → ArdışıkGün üzerinden özet

    Güçlendirilmiş:
    - analizli_kurum_tablolari tuple ise ilk elemanı (dict) alınır
    - tüm oran/pozitif/negatif/ilk5 hesapları yalnızca kurum kolonları üzerinden yapılır
    - ek kolonlar (DolaşımaOranı, Yorum vb.) hesaplamaya karışmaz
    """
    kolon_adi = "ArdışıkToplam" if tip == "AT" else "ArdışıkGün"

    # Eğer tuple geldiyse içinden dict'i al
    if isinstance(analizli_kurum_tablolari, tuple) and len(analizli_kurum_tablolari) > 0:
        analizli_kurum_tablolari = analizli_kurum_tablolari[0]

    # güvenlik: artık dict bekliyoruz
    if not isinstance(analizli_kurum_tablolari, dict):
        raise ValueError("analizli_kurum_tablolari bir dict olmalı (veya (dict, ... ) tuple).")

    # 1) Ana ozet_matrix: yalnızca kurum kolonları (hisse index'e göre align olur)
    ozet_matrix = pd.DataFrame()
    kurum_cols = []
    for kurum, df in analizli_kurum_tablolari.items():
        if kolon_adi in df.columns:
            # seri al, numeric'e çevir, index hisse olacak
            s = pd.to_numeric(df[kolon_adi], errors='coerce')
            ozet_matrix[kurum] = s
            kurum_cols.append(kurum)

    # Eğer hiç kurum yoksa boş DF dön
    if ozet_matrix.empty:
        return ozet_matrix

    # Kurum kolonlarını belirle (güvenli sıra)
    kurum_cols = [c for c in kurum_cols if c in ozet_matrix.columns]

    # 2) Temiz numeric DataFrame (yalnızca kurum kolonları)
    kurum_df = ozet_matrix[kurum_cols].apply(pd.to_numeric, errors='coerce')

    # 3) Max değerler (alıcı: pozitif max, satıcı: negatif en büyük mutlak -> negatif sayı olarak)
    max_alici_series = kurum_df.apply(lambda r: (r[r > 0].max() if (r[r > 0].any()) else np.nan), axis=1)
    max_satici_series = kurum_df.apply(lambda r: (-abs(r[r < 0]).max() if (r[r < 0].any()) else np.nan), axis=1)

    # 4) İlk 5 ve max kurum isimleri
    def top_k_list_pos(r, k=5):
        vals = r[r > 0].dropna()
        if vals.empty:
            return []
        return vals.sort_values(ascending=False).head(k).index.tolist()

    def top_k_list_neg(r, k=5):
        vals = r[r < 0].dropna()
        if vals.empty:
            return []
        # sıralama en çok satan (mutlak büyük) öne gelsin: küçük (negatif) değerden başlayarak
        return vals.sort_values().head(k).index.tolist()

    alici_first5 = kurum_df.apply(lambda r: top_k_list_pos(r, 5), axis=1)
    satici_first5 = kurum_df.apply(lambda r: top_k_list_neg(r, 5), axis=1)

    # 5) Max kurum isimleri
    def max_alici_kurum_name(r):
        pos = r[r > 0].dropna()
        return pos.idxmax() if not pos.empty else np.nan

    def max_satici_kurum_name(r):
        neg = r[r < 0].dropna()
        return neg.abs().idxmax() if not neg.empty else np.nan

    max_alici_kurum = kurum_df.apply(lambda r: max_alici_kurum_name(r), axis=1)
    max_satici_kurum = kurum_df.apply(lambda r: max_satici_kurum_name(r), axis=1)

    # 6) KurumAl_% / KurumSat_% (kurum sayısına göre)
    def calc_kurum_percentages(row):
        pozitifler = row[row > 0].dropna()
        negatifler = row[row < 0].dropna()
        al_count = len(pozitifler)
        sat_count = len(negatifler)
        toplam_count = al_count + sat_count
        if toplam_count == 0:
            return pd.Series([np.nan, np.nan])
        return pd.Series([round(al_count / toplam_count * 100, 2),
                          round(sat_count / toplam_count * 100, 2)])

    kurum_pct_df = kurum_df.apply(calc_kurum_percentages, axis=1)
    kurum_al_pct = kurum_pct_df.iloc[:, 0]
    kurum_sat_pct = kurum_pct_df.iloc[:, 1]

    # 7) Max kurumların ek bilgilerini (satır bazında döngü)
    alici_extra_main = []
    alici_extra_DO = []
    alici_extra_AG = []

    satici_extra_DO = []
    satici_extra_AG = []

    toplam_alici_DO = []
    toplam_satici_DO = []

    for idx in kurum_df.index:
        # Max alıcı kurum bilgiler
        ak = max_alici_kurum.get(idx, np.nan)
        if pd.notna(ak) and ak in analizli_kurum_tablolari:
            dfk = analizli_kurum_tablolari[ak]
            if idx in dfk.index:
                alici_extra_main.append(dfk.loc[idx, kolon_adi])
                alici_extra_DO.append(dfk.loc[idx, "DolaşımaOranı"] if "DolaşımaOranı" in dfk.columns else np.nan)
                alici_extra_AG.append(dfk.loc[idx, "ArdışıkGetiri"] if "ArdışıkGetiri" in dfk.columns else np.nan)
            else:
                alici_extra_main.append(np.nan); alici_extra_DO.append(np.nan); alici_extra_AG.append(np.nan)
        else:
            alici_extra_main.append(np.nan); alici_extra_DO.append(np.nan); alici_extra_AG.append(np.nan)

        # Max satıcı kurum bilgiler
        sk = max_satici_kurum.get(idx, np.nan)
        if pd.notna(sk) and sk in analizli_kurum_tablolari:
            dfs = analizli_kurum_tablolari[sk]
            if idx in dfs.index:
                satici_extra_DO.append(dfs.loc[idx, "DolaşımaOranı"] if "DolaşımaOranı" in dfs.columns else np.nan)
                satici_extra_AG.append(dfs.loc[idx, "ArdışıkGetiri"] if "ArdışıkGetiri" in dfs.columns else np.nan)
            else:
                satici_extra_DO.append(np.nan); satici_extra_AG.append(np.nan)
        else:
            satici_extra_DO.append(np.nan); satici_extra_AG.append(np.nan)

        # Toplam DO hesapla (yalnızca kurum_df üzerinden hangi kurumlar pozitif/negatif, ardından ilgili DO'ları topla)
        do_al = 0.0
        do_sat = 0.0
        for kurum, df in analizli_kurum_tablolari.items():
            if idx in df.index and kolon_adi in df.columns:
                val = df.loc[idx, kolon_adi]
                if pd.notna(val) and val != 0:
                    if val > 0 and "DolaşımaOranı" in df.columns:
                        do_al += df.loc[idx, "DolaşımaOranı"] or 0.0
                    elif val < 0 and "DolaşımaOranı" in df.columns:
                        do_sat += df.loc[idx, "DolaşımaOranı"] or 0.0
        toplam_alici_DO.append(do_al if do_al != 0 else np.nan)
        toplam_satici_DO.append(do_sat if do_sat != 0 else np.nan)

    # 8) Sonuç DataFrame: mevcut kurum_df (kurum kolonları) üzerine ek sütunlar
    result = kurum_df.copy()

    result[f"{kolon_adi}_Max_Alıcı"] = max_alici_series
    result[f"{kolon_adi}_Max_Satıcı"] = max_satici_series

    result[f"{kolon_adi}_Max_Alıcı_Kurum"] = max_alici_kurum
    result[f"{kolon_adi}_Max_Satıcı_Kurum"] = max_satici_kurum

    result[f"Alıcı_{kolon_adi}"] = alici_extra_main
    result["Alıcı_DolaşımaOranı"] = alici_extra_DO
    result["Alıcı_ArdışıkGetiri"] = alici_extra_AG

    result["Satıcı_DolaşımaOranı"] = satici_extra_DO
    result["Satıcı_ArdışıkGetiri"] = satici_extra_AG

    result["Toplam_Alıcı_DolaşımaOranı"] = toplam_alici_DO
    result["Toplam_Satıcı_DolaşımaOranı"] = toplam_satici_DO

    result["KurumAl_%"] = kurum_al_pct
    result["KurumSat_%"] = kurum_sat_pct

    # Yorum -> ilk 5 bilgilerini stringe çevirip ekle
    yorum_ser = alici_first5.combine(satici_first5, lambda a, s: f"{', '.join(a)} / {', '.join(s)}")
    result["Yorum"] = yorum_ser

    return result

def kurum_matrisleri_excele_kaydet(kurum_matrisleri, klasor_yolu,
                                   dosya_adi="kurum_zaman_serisi.xlsx",
                                   tarih_formati='%d/%m/%Y',
                                   max_tarih_sayisi=30,
                                   fiili_dolasim_dict=None,
                                   ozet_sheet_adi="Ozet"):

    dosya_yolu = os.path.join(klasor_yolu, dosya_adi)

    basarili_kurum = 0
    basarisiz_kurum = 0
    analizli_kurum_tablolari = {}

    with pd.ExcelWriter(dosya_yolu, engine='openpyxl') as writer:
        for kurum, matris_df in kurum_matrisleri.items():
            try:
                sheet_adi = kurum[:31] if len(kurum) > 31 else kurum

                matris_analizli = kurum_matrisi_analizli(
                    matris_df,
                    max_tarih_sayisi=max_tarih_sayisi,
                    fiili_dolasim_dict=fiili_dolasim_dict
                )
                analizli_kurum_tablolari[kurum] = matris_analizli

                formatted_columns = [
                    col.strftime(tarih_formati) if isinstance(col, pd.Timestamp) else str(col)
                    for col in matris_analizli.columns
                ]
                matris_analizli.columns = formatted_columns

                matris_analizli.to_excel(writer, sheet_name=sheet_adi)

                basarili_kurum += 1

            except Exception as e:
                print(f"✗ HATA ({kurum}): {e}")
                basarisiz_kurum += 1
                continue

        # --- ÖZET SHEET ---
        if analizli_kurum_tablolari:

            ozet_AT = kurum_ozet_matrisi_olustur_v2(analizli_kurum_tablolari, tip="AT")
            ozet_AG = kurum_ozet_matrisi_olustur_v2(analizli_kurum_tablolari, tip="AG")

            ozet_AT.to_excel(writer, sheet_name="ozet_ArdışıkToplam")
            ozet_AG.to_excel(writer, sheet_name="ozet_ArdışıkGün")

            # # Aynı sheet'e yaz,alt alta
            # ozet_AT.to_excel(writer, sheet_name=ozet_sheet_adi, startrow=0)
            # ozet_AG.to_excel(writer, sheet_name=ozet_sheet_adi, startrow=len(ozet_AT)+3)

            print(f"\n✓ Özet sheet'ler: ozet_AT ve ozet_AG eklendi.")



    print(f"\n✓ {basarili_kurum} kurum başarıyla kaydedildi, {basarisiz_kurum} kurum hatalı.")
    return analizli_kurum_tablolari, dosya_yolu



def kurum_ozet_istatistikleri(kurum_matrisleri):
    """
    Kurum bazlı özet istatistikler üretir.

    Parameters:
    -----------
    kurum_matrisleri : dict
        {kurum_adi: DataFrame} formatında sözlük

    Returns:
    --------
    DataFrame: Kurum özet istatistikleri
    """

    ozet_liste = []

    for kurum, matris_df in kurum_matrisleri.items():
        ozet = {
            'Kurum': kurum,
            'Toplam Hisse Sayısı': len(matris_df),
            'Toplam Gün Sayısı': len(matris_df.columns),
            'Ortalama Günlük İşlem': matris_df.notna().sum().mean(),
            'Toplam Net Pozisyon (Son Gün)': matris_df.iloc[:, -1].sum() if len(matris_df.columns) > 0 else 0,
            'En Aktif Hisse': matris_df.notna().sum(axis=1).idxmax() if len(matris_df) > 0 else None
        }
        ozet_liste.append(ozet)

    return pd.DataFrame(ozet_liste)


def hisse_bazli_kurum_karsilastirma(kurum_matrisleri, hisse_sembolu):
    """
    Belirli bir hisse için tüm kurumların zaman serisini karşılaştırır.

    Parameters:
    -----------
    kurum_matrisleri : dict
        {kurum_adi: DataFrame} formatında sözlük
    hisse_sembolu : str
        Analiz edilecek hisse

    Returns:
    --------
    DataFrame: Kurum x Tarih matrisi (belirli hisse için)
    """

    hisse_verileri = {}

    for kurum, matris_df in kurum_matrisleri.items():
        if hisse_sembolu in matris_df.index:
            hisse_verileri[kurum] = matris_df.loc[hisse_sembolu]

    if not hisse_verileri:
        print(f"Uyarı: {hisse_sembolu} hiçbir kurumda bulunamadı.")
        return pd.DataFrame()

    # DataFrame'e dönüştür
    karsilastirma_df = pd.DataFrame(hisse_verileri).T

    return karsilastirma_df


# =============================================================================
# KULLANIM ÖRNEKLERİ
# =============================================================================

# Örnek 3: Belirli bir kurumun matrisine bak
# -----------------------------------------------------------------------------
# yapi_kredi_matris = kurum_matrisleri['YAPI KREDI']
# print(f"YAPI KREDI: {yapi_kredi_matris.shape[0]} hisse x {yapi_kredi_matris.shape[1]} gün")
# print(yapi_kredi_matris.head(10))


# Örnek 4: Kurum özet istatistikleri
# -----------------------------------------------------------------------------
# ozet_df = kurum_ozet_istatistikleri(kurum_matrisleri)
# print(ozet_df.sort_values('Toplam Hisse Sayısı', ascending=False).head(20))


# Örnek 5: Belirli bir hisse için kurumları karşılaştır
# -----------------------------------------------------------------------------
# isctr_karsilastirma = hisse_bazli_kurum_karsilastirma(kurum_matrisleri, 'ISCTR')
# print("\nISCTR hissesi - Kurum bazlı zaman serisi:")
# print(isctr_karsilastirma)


# Örnek 6: Belirli bir kurumun belirli tarihteki pozisyonları
# -----------------------------------------------------------------------------
# # YAPI KREDI'nin 09/10/2025 tarihindeki tüm pozisyonları
# yapi_kredi_matris = kurum_matrisleri['YAPI KREDI']
# tarih = pd.to_datetime('2025-10-09')
#
# if tarih in yapi_kredi_matris.columns:
#     gunluk_pozisyon = yapi_kredi_matris[tarih].dropna()
#     print(f"\nYAPI KREDI - 09/10/2025 pozisyonları:")
#     print(gunluk_pozisyon.sort_values(ascending=False).head(10))



**Sayfa Sıralaması ve Düzenleme**

In [ ]:
def bicimlendir_exceli(excel_path):
    """
    Verilen Excel dosyasındaki tüm sheetleri formatlar:
    - Başlık: kalın, beyaz yazı, mavi arka plan
    - Zebra desen: alternatif satır renklendirme
    - Sayı formatlama: integer / binlik ayraç
    - Otomatik sütun genişliği
    - Filtreleme ekler
    """

    wb = load_workbook(excel_path)

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        max_row = ws.max_row
        max_col = ws.max_column

        # 1. PROFESYONEL BAŞLIKLAR
        for col in range(1, max_col + 1):
            header_cell = ws.cell(row=1, column=col)
            header_cell.font = Font(name="Calibri", size=8, bold=True, color="FFFFFF")  # Beyaz kalın yazı
            header_cell.fill = PatternFill("solid", fgColor="4F81BD")  # Mavi arkaplan
            header_cell.alignment = Alignment(horizontal="center", vertical="center")

        # 2. ZEBRA DESEN & HIZLI OKUNURLUK
        fill_even = PatternFill("solid", fgColor="DCE6F1")  # Açık mavi
        fill_odd = PatternFill("solid", fgColor="FFFFFF")   # Beyaz

        for row in range(2, max_row + 1):
            for col in range(1, max_col + 1):
                ws.cell(row=row, column=col).fill = fill_even if row % 2 == 0 else fill_odd
                ws.cell(row=row, column=col).font = Font(name="Calibri", size=8)  # Tüm hücrelerde font

        # 3. AKILLI SAYI FORMATLAMA (_Genel sheet'leri ve özet sheetleri)
        cols = {str(ws.cell(1, col).value).upper().strip(): col for col in range(1, max_col + 1)}

        hisse_col = cols.get("HISSE") or cols.get("HİSSE")
        gun_col = (cols.get("ARDIŞIKGÜN") or cols.get("ARDISIKGUN") or
                   cols.get("ARDIŞIKGÜN_MAX_ALICI") or cols.get("ARDIŞIKTOPLAM_MAX_ALICI"))

        if hisse_col and gun_col and hisse_col < gun_col:
            for col_idx in range(hisse_col + 1, gun_col):
                max_len = 10  # Minimum genişlik

                for row in range(2, max_row + 1):
                    cell = ws.cell(row=row, column=col_idx)

                    # SIFIR ve BOŞ değerleri atla
                    if cell.value in (0, None, ""):
                        cell.value = ""
                        continue

                    # TAM SAYI FORMATI (Binlik ayraçlı)
                    if isinstance(cell.value, (int, float)):
                        cell.number_format = "#,##0"
                        cell.alignment = Alignment(horizontal="right")

                        formatted = "{:,.0f}".format(abs(int(cell.value)))
                        max_len = max(max_len, len(formatted))

                ws.column_dimensions[get_column_letter(col_idx)].width = max(8, max_len + 2)

        # 4. OTOMATİK BOYUT AYARLAMA (başlık ve veri)
        for col in ws.iter_cols():
            col_letter = get_column_letter(col[0].column)
            max_len = max(len(str(cell.value)) if cell.value not in (0, None, "") else 0 for cell in col)
            if max_len > 0:
                ws.column_dimensions[col_letter].width = min(max_len + 2, 30)

        # 5. PROFESYONEL FİLTRELEME
        ws.auto_filter.ref = ws.dimensions  # Tüm veri aralığına filtre

        # 6. B2 hücresinde freeze panes
        ws.freeze_panes = "B2"



    # İlk sıraya taşınacak sayfalar (istediğiniz sırayla)
    preferred_sheets = [
        'ozet_ArdışıkToplam',
        'ozet_ArdışıkGün',
        'Pgc(5)_Ozet',
        'Netlot(5)_Ozet'

    ]

    # Mevcut sayfa isimlerini al
    current_sheets = wb.sheetnames

    # Her tercih edilen sayfayı kontrol et ve mevcutsa en başa taşı
    for sheet_name_to_move in reversed(preferred_sheets): # Tersten taşıyarak doğru sırayı elde ederiz
        if sheet_name_to_move in current_sheets:
            # Sayfayı ilk konuma taşı
            # move_sheet(sheet, offset=0) sheet'i en başa taşır
            wb.move_sheet(wb[sheet_name_to_move], offset=-wb.sheetnames.index(sheet_name_to_move))




    # print(f"✅ {', '.join(preferred_sheets)} sayfaları {excel_path} dosyasının başına taşındı.")

    # 7. KAYDET
    wb.save(excel_path)
    print(f"✨ {excel_path} profesyonel şekilde formatlandı!")



In [ ]:
# 1. Parquet'i oku
PgcGunlukArsiv_df, _ = en_guncel_parquet_dosyasini_bul()

# 3. Matrisleri oluştur
kurum_matrisleri = kurum_bazli_zaman_serisi_matrisi(PgcGunlukArsiv_df, tarih_kolonu='Tarih')
# bugun = datetime.now()
#pgc dosya sonundaki tarih kullandım
tarih_dt_from_str = tarih # Corrected: tarih is already a datetime object
tarih_formatted = tarih_dt_from_str.strftime('%d%m%y')

# 4. Excel'e kaydet (4 analiz kolonu ile)+Ozet tablo için verileri hazırla.
# kurum_matrisleri_excele_kaydet(
#     kurum_matrisleri,
#     '/content/',
#     dosya_adi=f"HisseKurumSeriDurumlarKurumlar_{tarih_formatted.strftime('%d%m%y')}.xlsx",
#     max_tarih_sayisi=30,
#     fiili_dolasim_dict=fiili_dolasim_dict
# )

analizli_kurum_tablolari,dosya_yolu = kurum_matrisleri_excele_kaydet(
    kurum_matrisleri,
    '/content/',
    dosya_adi=f"HisseKurumSeriDurumlar_{tarih_formatted}.xlsx",
    max_tarih_sayisi=30,
    fiili_dolasim_dict=fiili_dolasim_dict
)

# Tüm pgc_ozet ve NetLot_Ozet pivot sheetlerini de yaz
with pd.ExcelWriter(dosya_yolu, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for sheet_adi, pivot_df in pivotlar.items():
        pivot_df = pivot_df.fillna(0)
        pivot_df.index.name = 'Hisse' # Index adını "Hisse" olarak değiştirdim,diğerleri aynı olsun formatta sorun omasın diye
        pivot_df.to_excel(writer, sheet_name=sheet_adi, index=True)

print("Tüm pivot sheet'leri başarıyla güncellendi.")

# Excel dosyasını formatla
bicimlendir_exceli(dosya_yolu)

# **Analizler**

In [ ]:
# Görmek istediğin zaman aç
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy import stats
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

class BrokerNetPositionAnalyzer:
    def __init__(self, excel_file_path):
        self.excel_file = excel_file_path
        self.broker_data = {}
        self.consolidated_data = None

    def load_broker_data(self):
        try:
            xl_file = pd.ExcelFile(self.excel_file)
            all_sheets = xl_file.sheet_names
            if len(all_sheets) <= 4: return False
            broker_sheets = all_sheets[4:]
            print(f"Bulunan aracı kurum sayfaları: {len(broker_sheets)}")
            for sheet in broker_sheets:
                try:
                    self.broker_data[sheet.strip()] = pd.read_excel(self.excel_file, sheet_name=sheet)
                    print(f"- {sheet.strip()}: {len(self.broker_data[sheet.strip()])} hisse")
                except: continue
            return len(self.broker_data) > 0
        except: return False

    def preprocess_data(self):
        consolidated_list = []
        for broker_name, data in self.broker_data.items():
            all_cols = data.columns.tolist()
            if 'Hisse' not in all_cols: continue
            hisse_idx = all_cols.index('Hisse')
            date_cols = all_cols[hisse_idx + 1:-4]
            if len(date_cols) == 0: continue
            try:
                date_cols = sorted(date_cols, key=lambda x: pd.to_datetime(x, format='%d/%m/%Y'))
            except: continue

            for idx, row in data.iterrows():
                if pd.isna(row['Hisse']): continue
                daily_positions = [float(row.get(dc, 0)) if not pd.isna(row.get(dc, 0)) else 0 for dc in date_cols]
                ardisik_toplam = float(row.get('ArdışıkToplam', 0)) if not pd.isna(row.get('ArdışıkToplam', 0)) else 0
                ardisik_gun = int(row.get('ArdışıkGün', 0)) if not pd.isna(row.get('ArdışıkGün', 0)) else 0
                dolasim_orani = float(row.get('DolaşımaOranı', row.get('DolaşımOranı', 0))) if not pd.isna(row.get('DolaşımaOranı', row.get('DolaşımOranı', 0))) else 0
                toplam_getiri = float(row.get('ArdışıkGetiri', 0)) if not pd.isna(row.get('ArdışıkGetiri', 0)) else 0

                if ardisik_gun < 0: islem_yonu, ardisik_gun_abs = 'SATIŞ', abs(ardisik_gun)
                elif ardisik_gun > 0: islem_yonu, ardisik_gun_abs = 'ALIŞ', ardisik_gun
                else: islem_yonu, ardisik_gun_abs = 'NÖTR', 0

                positions = np.array(daily_positions)
                consecutive_strength = abs(ardisik_toplam) / max(1, ardisik_gun_abs) if ardisik_gun_abs > 0 else 0

                if len(positions) >= 2:
                    x = np.arange(len(positions))
                    slope, _, correlation, _, _ = stats.linregress(x, positions)
                    recent_momentum = np.mean(positions[-3:]) if len(positions) >= 3 else positions[-1]
                    momentum_adjusted = recent_momentum / max(np.std(positions), 0.001)
                else: slope = correlation = recent_momentum = momentum_adjusted = 0

                consolidated_list.append({
                    'Broker': broker_name, 'Hisse': row['Hisse'], 'ArdışıkToplam': ardisik_toplam,
                    'ArdışıkGün': ardisik_gun, 'ArdışıkGünMutlak': ardisik_gun_abs,
                    'DolaşımOranı': dolasim_orani, 'ToplamGetiri': toplam_getiri, 'İşlemYönü': islem_yonu,
                    'DailyPositions': daily_positions, 'DateColumns': date_cols,
                    'TotalVolume': np.sum(np.abs(positions)), 'NetPosition': np.sum(positions),
                    'Volatility': np.std(positions) if len(positions) > 1 else 0,
                    'PositiveDays': np.sum(positions > 0), 'NegativeDays': np.sum(positions < 0),
                    'ZeroDays': np.sum(positions == 0), 'ConsecutiveStrength': consecutive_strength,
                    'TrendSlope': slope, 'TrendCorrelation': correlation,
                    'RecentMomentum': recent_momentum, 'MomentumAdjusted': momentum_adjusted,
                    'MaxSingleDayVolume': np.max(np.abs(positions)) if len(positions) > 0 else 0
                })

        self.consolidated_data = pd.DataFrame(consolidated_list)
        print(f"Toplam {len(self.consolidated_data)} kayıt hazırlandı")
        print(f"ALIŞ: {len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'ALIŞ'])}")
        print(f"SATIŞ: {len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'SATIŞ'])}")
        print(f"NÖTR: {len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'NÖTR'])}")

    def broker_strategy_analysis(self):
        print("\n=== ARACI KURUM STRATEJİ ANALİZİ ===")
        bs = self.consolidated_data.groupby('Broker').agg({
            'ArdışıkToplam': ['sum', 'mean', 'count'], 'ArdışıkGünMutlak': ['mean', 'max'],
            'NetPosition': 'sum', 'ConsecutiveStrength': 'mean'
        }).round(3)
        bs.columns = ['_'.join(c) for c in bs.columns]
        bs['Strategy'] = bs.apply(lambda r: 'Aggressive_Buyer' if r['NetPosition_sum'] > 0 and r['ArdışıkGünMutlak_mean'] > 3
                                  else 'Aggressive_Seller' if r['NetPosition_sum'] < 0 and r['ArdışıkGünMutlak_mean'] > 3
                                  else 'Long_Term_Player' if r['ArdışıkGünMutlak_mean'] > 5
                                  else 'Active_Trader' if r['ArdışıkToplam_count'] > 20 else 'Selective_Trader', axis=1)
        print(bs['Strategy'].value_counts())
        for s in bs['Strategy'].unique():
            print(f"{s}: {bs[bs['Strategy'] == s].index.tolist()}")
        return bs

    def smart_money_analysis(self):
        print("\n=== SMART MONEY ANALİZİ ===")
        sa = []
        for stock in self.consolidated_data['Hisse'].unique():
            sd = self.consolidated_data[self.consolidated_data['Hisse'] == stock]
            if len(sd) < 2: continue
            tb = sd[sd['İşlemYönü'] == 'ALIŞ']['ArdışıkToplam'].sum()
            ts = abs(sd[sd['İşlemYönü'] == 'SATIŞ']['ArdışıkToplam'].sum())
            nf = tb - ts
            ar = sd['ToplamGetiri'].mean()
            bb, sb, tot = len(sd[sd['İşlemYönü'] == 'ALIŞ']), len(sd[sd['İşlemYönü'] == 'SATIŞ']), len(sd)
            cs = abs(bb - sb) / tot if tot > 0 else 0
            tv = tb + ts
            ss = (nf * ar * cs) / tv if tv > 0 else 0
            sa.append({'Hisse': stock, 'NetFlow': nf, 'AvgReturn': ar, 'ConsensusScore': cs, 'SmartScore': ss})

        sdf = pd.DataFrame(sa).sort_values('SmartScore', ascending=False)
        print("Top 10 Smart Money:")
        print(sdf.head(10)[['Hisse', 'SmartScore', 'NetFlow', 'AvgReturn']])
        print("\nBottom 10:")
        print(sdf.tail(10)[['Hisse', 'SmartScore', 'NetFlow', 'AvgReturn']])
        return sdf

    def consecutive_success_analysis(self):
        print("\n=== ARDIŞIK İŞLEM BAŞARI ANALİZİ ===")
        self.consolidated_data['Success'] = self.consolidated_data.apply(
            lambda r: 1 if (r['İşlemYönü'] == 'ALIŞ' and r['ToplamGetiri'] > 0) or
                          (r['İşlemYönü'] == 'SATIŞ' and r['ToplamGetiri'] < 0) else 0, axis=1)
        sbd = self.consolidated_data.groupby('ArdışıkGünMutlak').agg({
            'Success': ['sum', 'count', 'mean'], 'ToplamGetiri': 'mean'
        }).round(3)
        sbd.columns = ['_'.join(c) for c in sbd.columns]
        sbd['SuccessRate'] = sbd['Success_mean'] * 100
        print("Başarı Oranları (min 3 işlem):")
        print(sbd[sbd['Success_count'] >= 3][['Success_count', 'SuccessRate', 'ToplamGetiri_mean']])

        brs = self.consolidated_data.groupby(['Broker', 'İşlemYönü']).agg({
            'Success': ['sum', 'count', 'mean'], 'ToplamGetiri': 'mean'
        }).round(3)
        brs.columns = ['_'.join(c) for c in brs.columns]
        brs['SuccessRate'] = brs['Success_mean'] * 100
        print("\nBroker Başarı (min 3 işlem, top 15):")
        print(brs[brs['Success_count'] >= 3].sort_values('SuccessRate', ascending=False)[
            ['Success_count', 'SuccessRate', 'ToplamGetiri_mean']].head(15))
        return sbd, brs

    def anomaly_detection(self):
        print("\n=== ANOMALİ TESPİT ===")
        f = self.consolidated_data[['ArdışıkToplam', 'ArdışıkGünMutlak', 'TotalVolume',
                                    'Volatility', 'ConsecutiveStrength', 'MaxSingleDayVolume']].fillna(0)
        iso = IsolationForest(contamination=0.05, random_state=42)
        self.consolidated_data['Anomaly'] = iso.fit_predict(f)
        a = self.consolidated_data[self.consolidated_data['Anomaly'] == -1]
        print(f"Anomali: {len(a)}")
        if len(a) > 0:
            print("\nTop 10 Anomali:")
            print(a.nlargest(10, 'ArdışıkToplam')[['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkToplam', 'ArdışıkGünMutlak']])
        return a

    def momentum_timing_analysis(self):
        print("\n=== MOMENTUM ANALİZ ===")
        vt = self.consolidated_data['ArdışıkToplam'].quantile(0.75)
        sm = self.consolidated_data[(self.consolidated_data['ArdışıkGünMutlak'] >= 3) &
                                   (abs(self.consolidated_data['ArdışıkToplam']) > vt)]
        print(f"Güçlü momentum: {len(sm)}")
        if len(sm) > 0:
            print("\nTop 10:")
            print(sm.nlargest(10, 'ConsecutiveStrength')[['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkToplam', 'ConsecutiveStrength']])
        return sm

    def sector_analysis(self):
        print("\n=== SEKTÖR ANALİZ ===")
        SEC = {'Banka': ['AKBNK', 'ISCTR', 'YKBNK', 'GARAN', 'HALKB'],
               'Enerji': ['TUPRS', 'PETKM'], 'Holding': ['KCHOL', 'SAHOL']}
        self.consolidated_data['Sector'] = self.consolidated_data['Hisse'].apply(
            lambda x: next((s for s, st in SEC.items() if x.upper() in st), 'Diğer'))
        ss = self.consolidated_data.groupby(['Sector', 'İşlemYönü']).agg({
            'ArdışıkToplam': ['sum', 'count'], 'ToplamGetiri': 'mean', 'Success': 'mean'
        }).round(3)
        ss.columns = ['_'.join(c) for c in ss.columns]
        print(ss)
        return ss

    def same_direction_consensus_analysis(self):
        print("\n=== KONSENSÜS ANALİZ ===")
        cd = []
        for stock in self.consolidated_data['Hisse'].unique():
            sd = self.consolidated_data[self.consolidated_data['Hisse'] == stock]
            if len(sd) < 2: continue
            buyers = sd[sd['İşlemYönü'] == 'ALIŞ']
            sellers = sd[sd['İşlemYönü'] == 'SATIŞ']
            tot = len(sd)
            dc = max(len(buyers), len(sellers)) / tot if tot > 0 else 0
            cd.append({'Hisse': stock, 'TotalBrokers': tot, 'DominantConsensus': dc,
                      'BuyerList': buyers['Broker'].tolist(), 'SellerList': sellers['Broker'].tolist(),
                      'NetFlow': buyers['ArdışıkToplam'].sum() - abs(sellers['ArdışıkToplam'].sum()),
                      'AvgReturn': sd['ToplamGetiri'].mean()})

        cdf = pd.DataFrame(cd)
        sc = cdf[cdf['DominantConsensus'] >= 0.7].sort_values('DominantConsensus', ascending=False)
        print("GÜÇLÜ KONSENSÜS (70%+):")
        for _, r in sc.head(10).iterrows():
            dir = "ALIŞ" if len(r['BuyerList']) > len(r['SellerList']) else "SATIŞ"
            print(f"{r['Hisse']}: {dir} %{r['DominantConsensus']*100:.0f}")
            print(f"  Net: {r['NetFlow']:,.0f}, Getiri: {r['AvgReturn']:.2f}%\n")
        return cdf

    def broker_correlation_analysis(self):
        print("\n=== KORELASYON ===")
        p = self.consolidated_data.pivot_table(index='Hisse', columns='Broker', values='ArdışıkToplam', fill_value=0)
        if p.shape[1] < 2: return None
        bc = p.corr()
        corr = []
        for i in range(len(bc.columns)):
            for j in range(i+1, len(bc.columns)):
                if not pd.isna(bc.iloc[i,j]):
                    corr.append({'K1': bc.columns[i], 'K2': bc.columns[j], 'Corr': bc.iloc[i,j]})
        cdf = pd.DataFrame(corr).sort_values('Corr', ascending=False)
        print("Top 10 Pozitif:")
        print(cdf.head(10))
        print("\nTop 10 Negatif:")
        print(cdf.tail(10))
        return cdf, bc

    def success_prediction_analysis(self):
        print("\n=== ML TAHMİN ===")
        fc = ['ArdışıkGünMutlak', 'TotalVolume', 'ConsecutiveStrength', 'DolaşımOranı']
        self.consolidated_data['Broker_enc'] = pd.Categorical(self.consolidated_data['Broker']).codes
        self.consolidated_data['Dir_enc'] = self.consolidated_data['İşlemYönü'].map({'ALIŞ':1,'SATIŞ':-1,'NÖTR':0})
        fc.extend(['Broker_enc', 'Dir_enc'])
        X, y = self.consolidated_data[fc].fillna(0), self.consolidated_data['Success']
        if len(X) < 10 or y.sum() < 3: return None
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
        m = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        m.fit(Xtr, ytr)
        print(classification_report(yte, m.predict(Xte)))
        fi = pd.DataFrame({'F': fc, 'I': m.feature_importances_}).sort_values('I', ascending=False)
        print("\nTop Features:")
        print(fi.head(5))
        return m, fi

    def herd_behavior_analysis(self):
        print("\n=== SÜRÜ DAVRANIŞ ===")
        hd = []
        for stock in self.consolidated_data['Hisse'].unique():
            sd = self.consolidated_data[self.consolidated_data['Hisse'] == stock]
            if len(sd) < 2: continue
            bb = len(sd[sd['İşlemYönü'] == 'ALIŞ'])
            sb = len(sd[sd['İşlemYönü'] == 'SATIŞ'])
            tot = len(sd)
            hs = max(bb, sb) / tot if tot > 0 else 0
            hd.append({'Hisse': stock, 'TotalBrokers': tot, 'HerdScore': hs,
                      'Direction': 'ALIŞ' if bb > sb else 'SATIŞ'})
        hdf = pd.DataFrame(hd).sort_values('HerdScore', ascending=False)
        sh = hdf[hdf['HerdScore'] >= 0.8]
        print(f"Güçlü sürü: {len(sh)}")
        if len(sh) > 0:
            print(sh.head(10)[['Hisse', 'Direction', 'HerdScore', 'TotalBrokers']])
        return hdf

    def run_comprehensive_analysis(self):
        print("🚀 ANALİZ SİSTEMİ")
        print("="*50)
        if not self.load_broker_data(): return None
        self.preprocess_data()
        if self.consolidated_data is None or len(self.consolidated_data) == 0: return None

        results = {}
        try:
            results['strategy'] = self.broker_strategy_analysis()
            results['smart_money'] = self.smart_money_analysis()
            results['success'] = self.consecutive_success_analysis()
            results['anomaly'] = self.anomaly_detection()
            results['momentum'] = self.momentum_timing_analysis()
            results['sector'] = self.sector_analysis()
            results['consensus'] = self.same_direction_consensus_analysis()
            results['correlation'] = self.broker_correlation_analysis()
            results['prediction'] = self.success_prediction_analysis()
            results['herd'] = self.herd_behavior_analysis()

            print("\n✅ TÜM ANALİZLER TAMAMLANDI!")
            return results
        except Exception as e:
            print(f"❌ Hata: {e}")
            import traceback
            traceback.print_exc()
            return None

    def export_to_excel(self, results, output_file='HisseKurumAnaliz_raporu.xlsx'):
        """Tüm sonuçları Excel'e aktar"""
        analiz_dosya_adi=f"HisseKurumAnaliz_{tarih_formatted}.xlsx"
        print(f"\n📊 Excel raporu oluşturuluyor: {analiz_dosya_adi}")

        with pd.ExcelWriter(analiz_dosya_adi, engine='openpyxl') as writer:
            # 1. Özet Sayfa
            summary_data = {
                'Metrik': ['Toplam Kayıt', 'Broker Sayısı', 'Hisse Sayısı', 'Alış Pozisyonu', 'Satış Pozisyonu',
                          'Toplam Alış Hacmi', 'Toplam Satış Hacmi', 'Net Akış', 'Ortalama Getiri'],
                'Değer': [
                    len(self.consolidated_data),
                    self.consolidated_data['Broker'].nunique(),
                    self.consolidated_data['Hisse'].nunique(),
                    len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'ALIŞ']),
                    len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'SATIŞ']),
                    f"{self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'ALIŞ']['ArdışıkToplam'].sum():,.0f}",
                    f"{abs(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'SATIŞ']['ArdışıkToplam'].sum()):,.0f}",
                    f"{self.consolidated_data['ArdışıkToplam'].sum():,.0f}",
                    f"{self.consolidated_data['ToplamGetiri'].mean():.2f}%"
                ]
            }
            pd.DataFrame(summary_data).to_excel(writer, sheet_name='Özet', index=False)

            # 2. Smart Money
            if 'smart_money' in results and results['smart_money'] is not None:
                results['smart_money'].to_excel(writer, sheet_name='SmartMoney', index=False)

            # 3. Broker Stratejileri
            if 'strategy' in results and results['strategy'] is not None:
                results['strategy'].to_excel(writer, sheet_name='BrokerStratejiler')

            # 4. Konsensüs
            if 'consensus' in results and results['consensus'] is not None:
                results['consensus'].to_excel(writer, sheet_name='Konsensus', index=False)

            # 5. Sürü Davranışı
            if 'herd' in results and results['herd'] is not None:
                results['herd'].to_excel(writer, sheet_name='SuruDavranis', index=False)

            # 6. Korelasyon
            if 'correlation' in results and results['correlation'] is not None:
                if isinstance(results['correlation'], tuple):
                    results['correlation'][0].to_excel(writer, sheet_name='Korelasyon', index=False)

            # 7. Anomaliler
            if 'anomaly' in results and results['anomaly'] is not None and len(results['anomaly']) > 0:
                results['anomaly'][['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkToplam',
                                  'ArdışıkGünMutlak', 'ToplamGetiri']].to_excel(writer, sheet_name='Anomaliler', index=False)

            # 8. Tüm İşlemler
            self.consolidated_data[['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkToplam',
                                   'ArdışıkGünMutlak', 'ToplamGetiri', 'DolaşımOranı']].to_excel(
                writer, sheet_name='TumIslemler', index=False)

        print(f"✅ Excel raporu oluşturuldu: {output_file}")
        return output_file

    def print_summary(self, results):
        """Önemli bulguları konsola yazdır"""
        print("\n" + "="*60)
        print("📊 ÖNE ÇIKAN BULGULAR")
        print("="*60)

        # 1. Güçlü Sürü Davranışı
        if 'herd' in results and results['herd'] is not None:
            strong_herd = results['herd'][results['herd']['HerdScore'] >= 0.8]
            if len(strong_herd) > 0:
                print(f"\n🐑 SÜRÜ DAVRANIŞI:")
                print(f"  • {len(strong_herd)} hissede güçlü konsensüs (%80+)")
                print(f"  • Top 3: {', '.join(strong_herd.head(3)['Hisse'].tolist())}")

        # 2. En İyi Smart Money
        if 'smart_money' in results and results['smart_money'] is not None:
            top_smart = results['smart_money'].iloc[0]
            print(f"\n💰 SMART MONEY:")
            print(f"  • En yüksek skor: {top_smart['Hisse']}")
            print(f"    - Skor: {top_smart['SmartScore']:.3f}")
            print(f"    - Net Akış: {top_smart['NetFlow']:,.0f}")
            print(f"    - Getiri: {top_smart['AvgReturn']:.2f}%")

        # 3. En Başarılı Broker
        if 'success' in results and results['success'] is not None:
            broker_success = results['success'][1]  # İkinci element broker başarı tablosu
            if len(broker_success) > 0:
                top_broker = broker_success.sort_values('SuccessRate', ascending=False).iloc[0]
                print(f"\n🎯 EN BAŞARILI:")
                print(f"  • {top_broker.name[0]} - {top_broker.name[1]}")
                print(f"    - Başarı Oranı: {top_broker['SuccessRate']:.1f}%")
                print(f"    - İşlem Sayısı: {int(top_broker['Success_count'])}")

        # 4. Anomaliler
        if 'anomaly' in results and results['anomaly'] is not None:
            print(f"\n⚠️ ANOMALİLER:")
            print(f"  • {len(results['anomaly'])} anormal işlem tespit edildi")
            if len(results['anomaly']) > 0:
                top_anomaly = results['anomaly'].nlargest(1, 'ArdışıkToplam').iloc[0]
                print(f"  • En büyük: {top_anomaly['Broker']} - {top_anomaly['Hisse']}")
                print(f"    - Hacim: {top_anomaly['ArdışıkToplam']:,.0f}")

        # 5. Güçlü Konsensüs
        if 'consensus' in results and results['consensus'] is not None:
            strong_consensus = results['consensus'][results['consensus']['DominantConsensus'] >= 0.7]
            if len(strong_consensus) > 0:
                print(f"\n🤝 KONSENSÜS:")
                print(f"  • {len(strong_consensus)} hissede güçlü fikir birliği")
                top_cons = strong_consensus.iloc[0]
                print(f"  • En güçlü: {top_cons['Hisse']}")
                print(f"    - Konsensüs: {top_cons['DominantConsensus']*100:.0f}%")
                print(f"    - Net Akış: {top_cons['NetFlow']:,.0f}")

        print("\n" + "="*60)

In [ ]:
# KULLANIM
if __name__ == "__main__":
    if dosya_yolu:
        # Analizci oluştur ve çalıştır
        analyzer = BrokerNetPositionAnalyzer(dosya_yolu)
        results = analyzer.run_comprehensive_analysis()

        if results:
            # 1. Konsola özet yazdır
            analyzer.print_summary(results)

            # 2. Excel raporu oluştur
            analiz_dosya_adi=f"HisseKurumAnaliz_{tarih_formatted}.xlsx"
            excel_file = analyzer.export_to_excel(results, analiz_dosya_adi)

            print(f"\n🎯 SONUÇ: Analiz başarıyla tamamlandı!")
            print(f"📊 Excel Raporu: {excel_file}")
            print("📈 Yukarıdaki detaylı bulgulara göre aksiyon alabilirsiniz.")
        else:
            print("\n❌ Analiz tamamlanamadı.")
    else:
        print("\n❌ Dosya bulunamadı.")

In [ ]:

try:
    # Define a temporary directory to store files before zipping
    temp_dir = "/content/download_temp" # Explicitly set to /content
    os.makedirs(temp_dir, exist_ok=True)

    # Dosyaları geçici dizine kopyala

    shutil.copy(dosya_yolu, os.path.join(temp_dir, os.path.basename(dosya_yolu)))
    shutil.copy(yeni_parquetDosya, os.path.join(temp_dir, os.path.basename(yeni_parquetDosya)))
    shutil.copy(analiz_dosya_adi, os.path.join(temp_dir, os.path.basename(analiz_dosya_adi)))


    # zip dosya oluştur gecici dizinden
    zip_file_name = f"HisseKurumSeriDurumlar_{tarih_formatted}.zip"
    shutil.make_archive(zip_file_name.replace('.zip', ''), 'zip', temp_dir)

    # zip dosyasını indir
    files.download(zip_file_name)
    print(f"Tüm raporlar {zip_file_name} olarak başarıyla indirildi.")

    # Clean up temporary directory and zip file
    # shutil.rmtree(temp_dir)
    # os.remove(zip_file_name)

except Exception as e:
    print(f"Dosya indirme veya arşivleme hatası: {e}")

In [ ]:
shutil.rmtree(temp_dir)
os.remove(zip_file_name)

# **Bundan sonrakini çalıştırma**

**Matrisden nihai çözüm yapana kadar aşağıdaki dosyalardan üretimi gecici kullan**

In [ ]:
# tum_veriler[tum_veriler['Hisse']=='ADGYO'].tail(10)

In [ ]:
# # Kurum ve dosya adı eşleştirmesi
# kurum_eslesme = {
#     "AK": "Ak_", "BANK OF AMERICA": "Bofa_", "DENIZ": "Deniz_",
#     "GEDIK": "Gedik_", "HSBC": "Hsbc_", "IS": "Isyat_",
#     "MARBAS": "Marbas_", "QNB YATIRIM": "Qnb_", "TERA": "Tera_",
#     "YAPI KREDI": "Ykb_"
# }

# # pgc_latest_file'dan tarihi ve base_dir'i al
# tarih_parcasi = os.path.basename(pgc_latest_file).split('_')[1].split('.')[0]
# base_dir = os.path.dirname(os.path.abspath(pgc_latest_file))

# print(f"📅 Tarih: {tarih_parcasi}")
# print(f"📁 Kayıt dizini: {base_dir}")

# # Hisse index'ini sütuna dönüştür
# matris_df = matris.reset_index()

# # Hisse sütununun adını kontrol et
# hisse_sutun_adi = matris_df.columns[0]

# # Her kurum için Excel dosyası oluştur
# for kurum_adi, dosya_oneki in kurum_eslesme.items():
#     if kurum_adi in matris_df.columns:
#         try:
#             # Kurum verisini hazırla - Hisse ve kurum sütununu al
#             kurum_df = matris_df[[hisse_sutun_adi, kurum_adi]].copy()

#             # Sütun adlarını değiştir
#             kurum_df.columns = ['Hisse', 'Net (Adet)']

#             # NaN ve sıfır değerleri temizle
#             kurum_df = kurum_df.dropna(subset=['Net (Adet)'])
#             kurum_df = kurum_df[kurum_df['Net (Adet)'] != 0]

#             if not kurum_df.empty:
#                 # Hisse A-Z'ye sırala
#                 kurum_df = kurum_df.sort_values('Hisse')

#                 # Dosya adını oluştur
#                 dosya_adi = f"{dosya_oneki}{tarih_parcasi}.xlsx"
#                 dosya_yolu = os.path.join(base_dir, dosya_adi)

#                 # Excel'e kaydet
#                 with pd.ExcelWriter(dosya_yolu, engine='openpyxl') as writer:
#                     kurum_df.to_excel(writer, sheet_name='Rapor', index=False)

#                     # Sayfayı formatla
#                     worksheet = writer.sheets['Rapor']
#                     worksheet.column_dimensions['A'].width = 15  # Hisse
#                     worksheet.column_dimensions['B'].width = 15  # Net (Adet)

#                     # Başlıkları kalın yap
#                     for cell in worksheet[1]:
#                         cell.font = cell.font.copy(bold=True)

#                 print(f"✅ {kurum_adi:20} → {len(kurum_df):4} hisse → {dosya_adi}")
#             else:
#                 print(f"⚠️ {kurum_adi:20} → Veri yok (tüm değerler 0 veya NaN)")

#         except Exception as e:
#             print(f"❌ {kurum_adi:20} → Hata: {e}")
#     else:
#         print(f"🔴 {kurum_adi:20} → Matriste bulunamadı!")

# print(f"\n🎉 İşlem tamamlandı! Dosyalar '{base_dir}' dizinine kaydedildi.")

Ayrı ayrı excellerin olduğu Esas işlemlerin yapıldığı yer

In [ ]:
# # # 🔍 1️⃣ En güncel HisseKurumSeriDurumlar dosyasını bul,yukarıda alıyorum

# # 🔄 2️⃣ Yeni broker dosyalarını bulmak için önceki hali,daha sonra geçici bir süre yalnızca olanları okusun diye yaptım,nihayetinde matris den okuyacağım ve tüm kurumları alacağım.
# yeni_files = defaultdict(list)
# # for file in glob('/content/*_*.xlsx'):
# #     if 'Genel' not in file and 'HisseKurumSeriDurumlar' not in file:
# #         kurum = os.path.basename(file).split('_')[0]
# #         yeni_files[kurum].append(file)

# # print(f"Bulunan yeni broker dosyaları: {dict(yeni_files)}")

# izin_verilen_onEkler = ['Ak_', 'Bofa_', 'Deniz_', 'Gedik_', 'Hsbc_', 'Isyat_', 'Marbas_', 'Qnb_', 'Tera_', 'Ykb_']

# for file in glob('/content/*_*.xlsx'):
#     dosya_adi = os.path.basename(file)

#     # SADECE izin verilen öneklerle başlayanları al
#     if any(dosya_adi.startswith(prefix) for prefix in izin_verilen_onEkler):
#         kurum = dosya_adi.split('_')[0]
#         yeni_files[kurum].append(file)

# print(f"Bulunan yeni broker dosyaları: {dict(yeni_files)}")



# # 🔧 3️⃣ Mevcut tüm sheet'leri oku
# all_existing_sheets = {}
# try:
#     wb_temp = load_workbook(hissemaster_path)
#     for sheet_name in wb_temp.sheetnames:
#         all_existing_sheets[sheet_name] = pd.read_excel(hissemaster_path, sheet_name=sheet_name)
#     wb_temp.close()
#     print(f"Mevcut tüm sheet'ler okundu: {list(all_existing_sheets.keys())}")
# except Exception as e:
#     print(f"Mevcut sheet'leri okuma hatası: {e}")

# def dolasima_orani_hesapla(hisse_kodu, ardisik_toplam):
#     fiili_dolasim = fiili_dolasim_dict.get(hisse_kodu, 0)
#     if fiili_dolasim > 0:
#         return round((ardisik_toplam / fiili_dolasim) * 100, 2)
#     return 0.00

# # 🔧 4️⃣ Yeni dosyalar varsa ilgili broker sheet'lerini güncelle
# if yeni_files:
#     # Excel dosyasını workbook olarak aç
#     wb = load_workbook(hissemaster_path)

#     for kurum, dosyalar in yeni_files.items():
#         dosyalar = sorted(dosyalar)
#         sheet_name = f"{kurum}_Genel"

#         # Mevcut sheet'i oku (varsa)
#         try:
#             if sheet_name in all_existing_sheets:
#                 master_df = all_existing_sheets[sheet_name].set_index('Hisse')
#                 print(f"{kurum}: Mevcut sheet bulundu, güncelleniyor...")
#             else:
#                 master_df = pd.DataFrame()
#                 print(f"{kurum}: Yeni sheet oluşturuluyor...")
#         except:
#             master_df = pd.DataFrame()

#         hisse_set = set(master_df.index.tolist())
#         tarih_dosya = {}

#         # Yeni dosyalardan verileri oku
#         for file in dosyalar:
#             tarih_raw = os.path.basename(file).split('_')[-1].replace('.xlsx', '')
#             tarih = f"{tarih_raw[:2]}/{tarih_raw[2:]}/2025"
#             df = pd.read_excel(file)[['Hisse', 'Net (Adet)']].set_index('Hisse')
#             df = df[df.index.str.len() < 6]
#             hisse_set.update(df.index.tolist())
#             tarih_dosya[tarih] = df['Net (Adet)']

#         # Master dataframe'i güncelle
#         tum_hisseler = pd.Series(sorted(hisse_set), name='Hisse')
#         master_df = master_df.reindex(tum_hisseler).fillna(0)

#         # Yeni tarihleri ekle
#         for tarih, net_series in sorted(tarih_dosya.items()):
#             master_df[tarih] = master_df.index.map(net_series).fillna(0)

#         # Tarih kolonlarını sırala
#         tarih_kolonlari = [col for col in master_df.columns if '/' in col and len(col.split('/')) == 3]
#         tarih_kolonlari = sorted(
#             tarih_kolonlari,
#             key=lambda x: (int(x.split('/')[2]), int(x.split('/')[1]), int(x.split('/')[0]))
#         )

#         # Maksimum 30 tarih sütunu sınırı
#         if len(tarih_kolonlari) > 30:
#             fazla_sayisi = len(tarih_kolonlari) - 30
#             silinecekler = tarih_kolonlari[:fazla_sayisi]  # en eski tarihler
#             master_df.drop(columns=silinecekler, inplace=True)
#             tarih_kolonlari = tarih_kolonlari[fazla_sayisi:]  # kalan 30 tarihi tut
#             print(f"{sheet_name}: {fazla_sayisi} eski tarih sütunu silindi -> {silinecekler}")




#         def ard_ardina_gun(df_row):
#             values = df_row[tarih_kolonlari].fillna(0).astype(float).tolist()
#             ard_pos, ard_neg = 0, 0
#             for v in reversed(values):
#                 if v == 0: break
#                 if v > 0:
#                     if ard_neg > 0: break
#                     ard_pos += 1
#                 elif v < 0:
#                     if ard_pos > 0: break
#                     ard_neg += 1
#             if ard_pos > 0:
#                 return f"+{ard_pos}"
#             elif ard_neg > 0:
#                 return f"-{ard_neg}"
#             else:
#                 return "0"

#         def ard_ardina_toplam(df_row):
#             values = df_row[tarih_kolonlari].fillna(0).astype(float).tolist()
#             toplam = 0
#             ard_pos, ard_neg = 0, 0
#             for v in reversed(values):
#                 if v == 0: break
#                 if v > 0:
#                     if ard_neg > 0: break
#                     ard_pos += 1
#                     toplam += v
#                 elif v < 0:
#                     if ard_pos > 0: break
#                     ard_neg += 1
#                     toplam += v
#             return toplam

#         # Hesaplamaları yap
#         master_df = master_df[master_df.index.str.len() < 6]
#         master_df['ArdışıkToplam'] = master_df.apply(ard_ardina_toplam, axis=1)
#         master_df['ArdışıkGün'] = master_df.apply(ard_ardina_gun, axis=1)
#         master_df['DolaşımaOranı'] = master_df.apply(
#             lambda row: dolasima_orani_hesapla(row.name, row['ArdışıkToplam']),
#             axis=1
#         )

#         # Sütun sıralaması
#         final_columns = tarih_kolonlari + ['ArdışıkToplam', 'ArdışıkGün', 'DolaşımaOranı']
#         master_df = master_df[final_columns]

#         # Sheet'i güncelle
#         master_df_reset = master_df.reset_index()

#         # Eğer sheet varsa sil
#         if sheet_name in wb.sheetnames:
#             wb.remove(wb[sheet_name])

#         # Yeni sheet oluştur
#         ws = wb.create_sheet(sheet_name)

#         # Verileri yaz
#         for row_idx, row in enumerate(dataframe_to_rows(master_df_reset, index=False, header=True)):
#             for col_idx, value in enumerate(row, start=1):
#                 ws.cell(row=row_idx + 1, column=col_idx, value=value)

#         print(f"{kurum}: {sheet_name} sheet güncellendi.")


#     # Diğer mevcut broker sheet'lerine DolaşımaOranı'yı her zaman güncelle
#     for sheet_name, existing_df in all_existing_sheets.items():
#         if sheet_name.endswith('_Genel') and sheet_name not in [f"{kurum}_Genel" for kurum in yeni_files.keys()]:
#             print(f"\n{sheet_name} için DolaşımaOranı güncelleniyor...")
#             try:
#                 existing_df = existing_df.set_index('Hisse')
#                 if 'ArdışıkToplam' not in existing_df.columns:
#                     print(f"{sheet_name}: 'ArdışıkToplam' sütunu bulunamadı, DolaşımaOranı hesaplanamadı.")
#                     continue

#                 # Güncelle
#                 existing_df['DolaşımaOranı'] = existing_df.apply(
#                     lambda row: dolasima_orani_hesapla(row.name, row['ArdışıkToplam']),
#                     axis=1
#                 )

#                 existing_df_reset = existing_df.reset_index()

#                 # Sheet'i güncelle
#                 if sheet_name in wb.sheetnames:
#                     wb.remove(wb[sheet_name])

#                 ws = wb.create_sheet(sheet_name)

#                 for row_idx, row in enumerate(dataframe_to_rows(existing_df_reset, index=False, header=True)):
#                     for col_idx, value in enumerate(row, start=1):
#                         ws.cell(row=row_idx + 1, column=col_idx, value=value)

#                 print(f"{sheet_name}: DolaşımaOranı sütunu güncellendi.")
#             except Exception as e:
#                 print(f"{sheet_name} için DolaşımaOranı güncellenemedi: {e}")



#     # Workbook'u kaydet
#     wb.save(hissemaster_path)
#     wb.close()

# else:
#     print("Yeni broker dosyası bulunamadı ve FiiliDolasim değişmedi, mevcut dosya korunuyor.")


# # Yeni dosya olmasa bile FiiliDolasim güncellenmişse tüm broker sheet'lerini güncelle
# wb = load_workbook(hissemaster_path)
# sheet_names = wb.sheetnames

# for sheet_name in sheet_names:
#     if sheet_name.endswith('_Genel'):
#         kurum = sheet_name.replace('_Genel', '')
#         print(f"\n{sheet_name} için DolaşımaOranı güncelleniyor...")
#         try:
#             # Sheet'i pandas dataframe olarak oku
#             df = pd.read_excel(hissemaster_path, sheet_name=sheet_name)

#             # Eğer ArdışıkToplam sütunu yoksa atla
#             if 'ArdışıkToplam' not in df.columns:
#                 print(f"{sheet_name}: 'ArdışıkToplam' sütunu bulunamadı, DolaşımaOranı hesaplanamadı.")
#                 continue

#             # DolaşımaOranı'nı güncelle
#             df['DolaşımaOranı'] = df.apply(
#                 lambda row: dolasima_orani_hesapla(row['Hisse'], row['ArdışıkToplam']),
#                 axis=1
#             )

#             # Sheet'i workbook'tan sil
#             if sheet_name in wb.sheetnames:
#                 wb.remove(wb[sheet_name])

#             # Yeni sheet oluştur
#             ws = wb.create_sheet(sheet_name)

#             # Verileri yaz
#             for row_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
#                 for col_idx, value in enumerate(row, 1):
#                     ws.cell(row=row_idx, column=col_idx, value=value)

#             print(f"{sheet_name}: DolaşımaOranı sütunu güncellendi.")

#         except Exception as e:
#             print(f"{sheet_name} için DolaşımaOranı güncellenemedi: {e}")

# # Workbook'u kaydet
# wb.save(hissemaster_path)
# wb.close()



# def getiri_hesapla_ve_ekle(hissemaster_path, tum_veriler):
#     """
#     Excel'deki her '_Genel' sheetine, hisselerin belirtilen gün sayısı kadar geriye dönük getirisini hesaplayarak
#     'ToplamGetiri' sütunu ekler.

#     Parametreler:
#         hissemaster_path (str): İşlenecek Excel dosyasının yolu
#         tum_veriler (DataFrame): Tarih, Hisse ve degisim sütunlarını içeren DataFrame
#     """
#     try:
#         print("\n" + "="*50)
#         print("📊 Getiri hesaplamaları başlıyor...")

#         # Workbook'u aç
#         wb = load_workbook(hissemaster_path)

#         # Tarih parser fonksiyonu
#         def parse_excel_date(date_str):
#             """
#             Excel'deki farklı tarih formatlarını datetime'a çevirir
#             """
#             try:
#                 if isinstance(date_str, str):
#                     # DD/MM/YYYY, DD.MM.YYYY, DD-MM-YYYY formatlarını destekler
#                     date_str = date_str.replace('.', '/').replace('-', '/')
#                     return pd.to_datetime(date_str, dayfirst=True)
#                 return pd.to_datetime(date_str)
#             except:
#                 return pd.NaT

#         # Her sheet için işlem yap
#         for sheet_name in wb.sheetnames:
#             if not sheet_name.endswith('_Genel'):
#                 continue

#             try:
#                 print(f"\n🔹 {sheet_name} sheet'i işleniyor...")

#                 # Sheet'i oku
#                 df = pd.read_excel(hissemaster_path, sheet_name=sheet_name)

#                 # Gerekli sütun kontrolü
#                 required_columns = ['Hisse', 'ArdışıkGün']
#                 if not all(col in df.columns for col in required_columns):
#                     print(f"⚠️ Gerekli sütunlar eksik: {required_columns}")
#                     continue

#                 # Tarih sütunlarını bul (Hisse ve ArdışıkToplam hariç)
#                 date_columns = []
#                 for col in df.columns:
#                     if col in ['Hisse', 'ArdışıkGün', 'ArdışıkToplam']:
#                         continue
#                     date = parse_excel_date(col)
#                     if not pd.isna(date):
#                         date_columns.append((col, date))

#                 # Tarihleri yeniden eskiye sırala
#                 date_columns.sort(key=lambda x: x[1], reverse=True)

#                 #print(f"📅 İşlenecek tarih sayısı: {len(date_columns)}")
#                 if date_columns:
#                     print(f"   Örnek tarihler: {date_columns[0][0]} -> {date_columns[0][1].strftime('%d/%m/%Y')} ... {date_columns[-1][0]} -> {date_columns[-1][1].strftime('%d/%m/%Y')}")

#                 # Getiri hesaplama
#                 df['ToplamGetiri'] = 0.0  # Varsayılan değer

#                 for idx, row in df.iterrows():
#                     hisse = str(row['Hisse']).strip()

#                     try:
#                         gun_sayisi = abs(int(float(str(row['ArdışıkGün']).strip())))
#                     except:
#                         gun_sayisi = 0

#                     if gun_sayisi == 0:
#                         continue

#                     # Getiriyi hesapla
#                     total = 0.0
#                     used_dates = []
#                     for col_name, date in date_columns[:gun_sayisi]:
#                         matched = tum_veriler[
#                             (tum_veriler['Hisse'] == hisse) &
#                             (tum_veriler['tarih'] == date)
#                         ]

#                         if not matched.empty and not pd.isna(matched.iloc[0]['degisim']):
#                             total += matched.iloc[0]['degisim']
#                             used_dates.append(date.strftime('%d/%m'))

#                     # Yuvarla ve ata
#                     df.at[idx, 'ToplamGetiri'] = round(total, 2)

#                     # Debug için ilk 3 hisseyi göster
#                     if idx < 3:
#                         print(f"   🧮 {hisse} | {gun_sayisi}gün: {used_dates} -> Toplam: {round(total, 2)}")

#                 # Sheet'i güncelle
#                 if sheet_name in wb.sheetnames:
#                     wb.remove(wb[sheet_name])
#                 ws = wb.create_sheet(sheet_name)

#                 # Sütun sıralaması (ToplamGetiri en sağda)
#                 cols = [c for c in df.columns if c != 'ToplamGetiri'] + ['ToplamGetiri']
#                 df = df[cols]

#                 for r in dataframe_to_rows(df, index=False, header=True):
#                     ws.append(r)

#                 # Workbook'u kaydet (KRİTİK NOKTA)
#                 wb.save(hissemaster_path)
#                 print(f"✅ {sheet_name} başarıyla güncellendi")

#             except Exception as e:
#                 print(f"❌ {sheet_name} işlenirken hata: {str(e)}")
#                 continue

#         wb.close()
#         print("\n🎉 TÜM GETİRİ HESAPLAMALARI TAMAMLANDI!")
#         return True

#     except Exception as e:
#         print(f"\n❗ KRİTİK HATA: {str(e)}")
#         if 'wb' in locals(): wb.close()
#         return False


# # Getiri hesaplamayı çalıştır
# getiri_hesapla_ve_ekle(hissemaster_path, tum_veriler)
# print("✅ Getiri hesaplamaları tamamlandı")


# #🔄 5️⃣ Güncel PgcGunlukArsiv parquet dosyasından Hisse-Tarih bazlı genel 'Para_Giris_Cikis' Pivot tablosu oluştukdan sonra excele sheet olarak kaydet

# # Dosya yoksa oluştur
# if not os.path.exists(hissemaster_path):
#     print(f"⚠ Dosya bulunamadı: {hissemaster_path}")
#     Workbook().save(hissemaster_path)

# # openpyxl ile daha önce açılan workbook'u kapat
# try:
#     wb = load_workbook(hissemaster_path)
#     wb.close()
# except:
#     pass

# # Tüm pivot sheetlerini yaz
# with pd.ExcelWriter(hissemaster_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     for sheet_adi, pivot_df in pivotlar.items():
#         pivot_df = pivot_df.fillna(0)
#         pivot_df.index.name = 'Hisse' # Index adını "Hisse" olarak değiştirdim,diğerleri aynı olsun formatta sorun omasın diye
#         pivot_df.to_excel(writer, sheet_name=sheet_adi, index=True)

# print("Tüm pivot sheet'leri başarıyla güncellendi.")


# # 🔄 6️⃣ En büyük tarihli yeni dosya ismiyle kaydet
# tum_tarihler = []
# wb = load_workbook(hissemaster_path)
# for sheet in wb.sheetnames:
#     if not sheet.endswith('_Genel'):
#         continue
#     try:
#         df = pd.read_excel(hissemaster_path, sheet_name=sheet)
#         tarih_kolonlari = [col for col in df.columns if '/' in col and len(col.split('/')) == 3]
#         tum_tarihler.extend(tarih_kolonlari)
#     except Exception as e:
#         print(f"{sheet} sheet tarih okuma hatası: {e}")
# wb.close()

# tum_tarihler = list(set(tum_tarihler))
# if tum_tarihler:
#     tum_tarihler = sorted(
#         tum_tarihler,
#         key=lambda x: (int(x.split('/')[2]), int(x.split('/')[1]), int(x.split('/')[0]))
#     )
#     son_tarih = tum_tarihler[-1]
#     son_tarih_str = son_tarih.replace('/', '')
#     print(f"En son tarih: {son_tarih}")
# else:
#     son_tarih_str = "YOK"
#     print("Hiçbir tarih bulunamadı.")

# final_output = f"/content/HisseKurumSeriDurumlar_{son_tarih_str}.xlsx"

# # Eğer dosya adı zaten aynıysa rename etme
# if hissemaster_path != final_output:
#     os.rename(hissemaster_path, final_output)
#     print(f"Dosya yeniden adlandırıldı: {final_output}")
# else:
#     print(f"Dosya zaten doğru isimde: {final_output}")

# # try:
# #     files.download(final_output)
# #     print("Dosya başarıyla indirildi.")
# # except Exception as e:
# #     print(f"Dosya indirme hatası: {e}")


En Çok Ardışık Alıcı/Satıcı Kurumlar Özet Raporu

In [ ]:
# def alici_satici_ozet_raporu(excel_path):
#     wb = load_workbook(excel_path)
#     sheets = [s for s in wb.sheetnames if s.endswith('_Genel')]

#     # 1️⃣ Tüm verileri nested dict yapısında topla
#     data = {}  # {hisse: {kurum: {'ardisik': x, 'dolasim': y, 'getiri': z}}}

#     for sheet in sheets:
#         kurum = sheet.replace('_Genel', '')
#         df = pd.read_excel(excel_path, sheet_name=sheet).set_index('Hisse')

#         for hisse, row in df.iterrows():
#             ard_gun = str(row['ArdışıkGün']).strip()

#             # ArdışıkGün parse
#             if ard_gun in ['nan', '', 'None']:
#                 sayi = 0
#             elif ard_gun.startswith('+'):
#                 sayi = int(float(ard_gun[1:]))
#             elif ard_gun.startswith('-'):
#                 sayi = -int(float(ard_gun[1:]))
#             else:
#                 sayi = int(float(ard_gun))

#             if hisse not in data:
#                 data[hisse] = {}
#             data[hisse][kurum] = {
#                 'ardisik': sayi,
#                 'dolasim': row.get('DolaşımaOranı', None),
#                 'getiri': row.get('ToplamGetiri', None)
#             }

#     # 2️⃣ ArdışıkGün tablosunu oluştur
#     rapor_df = pd.DataFrame({h: {k: v['ardisik'] for k, v in kurums.items()} for h, kurums in data.items()}).T.fillna(0).astype(int)

#     # Max alıcı & satıcı hesapla
#     rapor_df['Max_Alıcı'] = rapor_df[rapor_df > 0].max(axis=1).fillna(0).astype(int)
#     rapor_df['Max_Satıcı'] = rapor_df[rapor_df < 0].min(axis=1).fillna(0).astype(int)

#     # 3️⃣ Alıcı / Satıcı kurumları bul ve ilgili sheet'ten verileri ekle
#     alis_dolasim, alis_getiri, satis_dolasim, satis_getiri = [], [], [], []
#     kurum_al_oranlari, kurum_sat_oranlari = [], []  # Yeni eklenen oran listeleri
#     tum_alis_dolasim, tum_satis_dolasim = [], []  # Yeni eklenen toplam dolaşım oranları

#     for hisse, row in rapor_df.iterrows():
#         max_alici = row['Max_Alıcı']
#         max_satici = row['Max_Satıcı']
#         hisse_data = data[hisse]

#         # Alıcı/Satıcı/Nötr kurum sayılarını hesapla
#         alicilar = [k for k, v in hisse_data.items() if v['ardisik'] > 0]
#         saticilar = [k for k, v in hisse_data.items() if v['ardisik'] < 0]
#         toplam_kurum = len(hisse_data)

#         # Tüm alıcı ve satıcıların dolaşım oranlarını topla
#         alicilar_dolasim_toplam = sum(v['dolasim'] for k, v in hisse_data.items()
#                                     if v['ardisik'] > 0 and v['dolasim'] is not None)
#         saticilar_dolasim_toplam = sum(v['dolasim'] for k, v in hisse_data.items()
#                                      if v['ardisik'] < 0 and v['dolasim'] is not None)

#         tum_alis_dolasim.append(round(alicilar_dolasim_toplam, 2)) if alicilar_dolasim_toplam != 0 else tum_alis_dolasim.append(None)
#         tum_satis_dolasim.append(round(saticilar_dolasim_toplam, 2)) if saticilar_dolasim_toplam != 0 else tum_satis_dolasim.append(None)

#         # Oranları hesapla - 0 ise boş geç (None)
#         alici_oran = round(len(alicilar)/toplam_kurum * 100, 1) if toplam_kurum > 0 and len(alicilar) > 0 else None
#         satici_oran = round(len(saticilar)/toplam_kurum * 100, 1) if toplam_kurum > 0 and len(saticilar) > 0 else None

#         kurum_al_oranlari.append(alici_oran)
#         kurum_sat_oranlari.append(satici_oran)

#         # 📝 Not: Eğer birden fazla kurumda aynı max değer varsa **ilk bulunanı** alıyoruz
#         if max_alici > 0:
#             alici_kurum = rapor_df.columns[(rapor_df.loc[hisse] == max_alici)][0]
#             alici_data = data[hisse].get(alici_kurum, {})
#             alis_dolasim.append(alici_data.get('dolasim'))
#             alis_getiri.append(alici_data.get('getiri'))
#         else:
#             alis_dolasim.append(None)
#             alis_getiri.append(None)

#         if max_satici < 0:
#             satici_kurum = rapor_df.columns[(rapor_df.loc[hisse] == max_satici)][0]
#             satici_data = data[hisse].get(satici_kurum, {})
#             satis_dolasim.append(satici_data.get('dolasim'))
#             satis_getiri.append(satici_data.get('getiri'))
#         else:
#             satis_dolasim.append(None)
#             satis_getiri.append(None)

#     rapor_df['Alan_DolaşımaOranı'] = alis_dolasim
#     rapor_df['Alan_ToplamGetiri'] = alis_getiri
#     rapor_df['Satan_DolaşımaOranı'] = satis_dolasim
#     rapor_df['Satan_ToplamGetiri'] = satis_getiri

#     # Yeni eklenen oran sütunları
#     rapor_df['KurumAl_%'] = kurum_al_oranlari
#     rapor_df['KurumSat_%'] = kurum_sat_oranlari

#     # Yeni eklenen toplam dolaşım oranları
#     rapor_df['Alanların_DolaşımaOranı'] = tum_alis_dolasim
#     rapor_df['Satanların_DolaşımaOranı'] = tum_satis_dolasim

#     # 4️⃣ YORUM sütunu oluştur (alıcılar/satıcılar/nötrleri gruplayarak)
#     yorumlar = []
#     for hisse in rapor_df.index:
#         hisse_data = data[hisse]

#         alicilar = [kurum for kurum, info in hisse_data.items() if info['ardisik'] > 0]
#         saticilar = [kurum for kurum, info in hisse_data.items() if info['ardisik'] < 0]
#         notrler = [kurum for kurum, info in hisse_data.items() if info['ardisik'] == 0]

#         parcalar = []
#         if alicilar:
#             parcalar.append(f"{', '.join(alicilar)} alıcılı")
#         if saticilar:
#             parcalar.append(f"{', '.join(saticilar)} satıcılı")
#         if notrler:
#             parcalar.append(f"{', '.join(notrler)} nötr")

#         yorumlar.append(" / ".join(parcalar))

#     rapor_df['Yorum'] = yorumlar

#     #5️⃣ Excel'e yaz
#     with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#         rapor_df.to_excel(writer, sheet_name='Ozet_AlıcıSatıcı')

#     print("✅ Alıcı/Satıcı raporu oluşturuldu (Alış & Satış detayları + yorum dahil).")

# #Fonksiyonu çağır
# alici_satici_ozet_raporu(final_output)

**Format,stil işlemleri**

In [ ]:
# def format_excel(excel_path):
#     wb = load_workbook(excel_path)

#     for sheet_name in wb.sheetnames:
#         ws = wb[sheet_name]
#         max_row = ws.max_row
#         max_col = ws.max_column

#         # 1. PROFESYONEL BAŞLIKLAR
#         for col in range(1, max_col + 1):
#             header_cell = ws.cell(row=1, column=col)
#             header_cell.font = Font(bold=True, color="FFFFFF")  # Beyaz kalın yazı
#             header_cell.fill = PatternFill("solid", fgColor="4F81BD")  # Mavi arkaplan
#             header_cell.alignment = Alignment(horizontal="center", vertical="center")

#         # 2. ZEBRA DESEN & HIZLI OKUNURLUK
#         fill_even = PatternFill("solid", fgColor="DCE6F1")  # Açık mavi
#         fill_odd = PatternFill("solid", fgColor="FFFFFF")   # Beyaz

#         for row in range(2, max_row + 1):
#             for col in range(1, max_col + 1):
#                 ws.cell(row=row, column=col).fill = fill_even if row % 2 == 0 else fill_odd

#         # 3. AKILLI SAYI FORMATLAMA (_Genel sheet'leri için)
#         #if sheet_name.endswith("_Genel"): #Daha önce bu koşul vardı
#         if sheet_name.endswith("_Genel") or sheet_name == f'Pgc({top_n})_Ozet' or sheet_name == f'Netlot({top_n})_Ozet':
#             # Dinamik sütun bulma (Türkçe/İngilizce karakter uyumlu)
#             cols = {str(ws.cell(1, col).value).upper().strip(): col
#                    for col in range(1, max_col + 1)}

#             hisse_col = cols.get("HISSE") or cols.get("HİSSE")
#             gun_col = cols.get("ARDIŞIKGÜN") or cols.get("ARDISIKGUN") or cols.get("GÜN") or cols.get("GUN")

#             if hisse_col and gun_col and hisse_col < gun_col:
#                 for col_idx in range(hisse_col + 1, gun_col):
#                     max_len = 10  # Minimum genişlik

#                     for row in range(2, max_row + 1):
#                         cell = ws.cell(row=row, column=col_idx)

#                         # SIFIR ve BOŞ değerleri atla
#                         if cell.value in (0, None, ""):
#                             cell.value = ""
#                             continue

#                         # TAM SAYI FORMATI (Binlik ayraçlı)
#                         if isinstance(cell.value, (int, float)):
#                             cell.number_format = "#,##0"
#                             cell.alignment = Alignment(horizontal="right")

#                             # Milyar ölçeğine göre genişlik ayarla
#                             formatted = "{:,.0f}".format(abs(int(cell.value)))
#                             max_len = max(max_len, len(formatted))

#                     # Sütun genişliğini optimize et (8-13 karakter arası)
#                     # ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len, 8), 13)
#                     # ws.column_dimensions[get_column_letter(col_idx)].width = max_len + 2
#                     ws.column_dimensions[get_column_letter(col_idx)].width = max(8, max_len + 2)


#         # 4. OTOMATİK BOYUT AYARLAMA
#         for col in ws.iter_cols():
#             col_letter = get_column_letter(col[0].column)

#             # Başlık + Verilerdeki max uzunluk
#             max_len = max(
#                 len(str(cell.value)) if cell.value not in (0, None, "") else 0
#                 for cell in col
#             )

#             if max_len > 0:
#                 ws.column_dimensions[col_letter].width = min(max_len + 2, 30)

#         # 5. PROFESYONEL FİLTRELEME
#         ws.auto_filter.ref = ws.dimensions  # Tüm veri aralığına filtre

#     # 6. MÜKEMMEL KAYDETME
#     wb.save(excel_path)
#     print(f"✨ {excel_path} profesyonel şekilde formatlandı!")



In [ ]:
# format_excel(final_output)

# try:
#     # Define a temporary directory to store files before zipping
#     temp_dir = "/content/download_temp" # Explicitly set to /content
#     os.makedirs(temp_dir, exist_ok=True)

#     # Move files to the temporary directory
#     shutil.copy(final_output, os.path.join(temp_dir, os.path.basename(final_output)))
#     shutil.copy(dosya_yolu, os.path.join(temp_dir, os.path.basename(dosya_yolu)))
#     shutil.copy(yeni_parquetDosya, os.path.join(temp_dir, os.path.basename(yeni_parquetDosya)))

#     # Create a zip archive of the temporary directory
#     zip_file_name = f"HisseKurumSeriDurumlar_{son_tarih_str}.zip"
#     shutil.make_archive(zip_file_name.replace('.zip', ''), 'zip', temp_dir)

#     # Download the single zip archive
#     files.download(zip_file_name)
#     print(f"Tüm raporlar {zip_file_name} olarak başarıyla indirildi.")

#     # Clean up temporary directory and zip file
#     # shutil.rmtree(temp_dir)
#     # os.remove(zip_file_name)

# except Exception as e:
#     print(f"Dosya indirme veya arşivleme hatası: {e}")

In [ ]:
# shutil.rmtree(temp_dir)
# os.remove(zip_file_name)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# **Analizler**  bazı excel dosyalarından olanla için

In [ ]:

# from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import RandomForestClassifier, IsolationForest
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report
# from scipy import stats
# from scipy.spatial.distance import pdist, squareform
# from scipy.stats import chi2_contingency
# import networkx as nx
# import warnings
# warnings.filterwarnings('ignore')

# class BrokerNetPositionAnalyzer:
#     def __init__(self, excel_file_path):
#         self.excel_file = excel_file_path
#         self.broker_data = {}
#         self.consolidated_data = None

#     def load_broker_data(self):
#         """Excel dosyasından _Genel ile biten sayfaları yükle"""
#         try:
#             xl_file = pd.ExcelFile(self.excel_file)
#             broker_sheets = [sheet for sheet in xl_file.sheet_names if sheet.endswith('_Genel')]

#             print(f"Bulunan aracı kurum sayfaları: {len(broker_sheets)}")
#             for sheet in broker_sheets:
#                 broker_name = sheet.replace('_Genel', '')
#                 self.broker_data[broker_name] = pd.read_excel(self.excel_file, sheet_name=sheet)
#                 print(f"- {broker_name}: {len(self.broker_data[broker_name])} hisse")

#         except Exception as e:
#             print(f"Dosya okuma hatası: {e}")
#             return False
#         return True

#     def preprocess_data(self):
#         """Veriyi analiz için hazırla - ArdışıkGün negatif değerleri doğru işle"""
#         consolidated_list = []

#         for broker_name, data in self.broker_data.items():
#             # Tarih sütunlarını belirle
#             date_cols = []
#             start_found = False

#             for col in data.columns:
#                 if col == 'Hisse':
#                     start_found = True
#                     continue
#                 elif col == 'ArdışıkToplam':
#                     break
#                 elif start_found:
#                     date_cols.append(col)

#             # Tarihleri sırala
#             date_cols = sorted(date_cols, key=lambda x: pd.to_datetime(x, format='%d/%m/%Y'))

#             # Her hisse için veri hazırla
#             for idx, row in data.iterrows():
#                 if pd.isna(row['Hisse']):
#                     continue

#                 # Günlük net pozisyonlar
#                 daily_positions = []
#                 for date_col in date_cols:
#                     value = row.get(date_col, 0)
#                     if pd.isna(value):
#                         value = 0
#                     daily_positions.append(float(value))

#                 # Ardışık bilgileri - NEGATİF GÜN = SATIŞ yönü
#                 ardisik_toplam = row.get('ArdışıkToplam', 0)
#                 ardisik_gun = row.get('ArdışıkGün', 0)
#                 dolasim_orani = row.get('DolaşımaOranı', 0)
#                 toplam_getiri = row.get('ToplamGetiri', 0)

#                 # NaN değerleri temizle
#                 if pd.isna(ardisik_toplam): ardisik_toplam = 0
#                 if pd.isna(ardisik_gun): ardisik_gun = 0
#                 if pd.isna(dolasim_orani): dolasim_orani = 0
#                 if pd.isna(toplam_getiri): toplam_getiri = 0

#                 # İşlem yönünü ArdışıkGün sütunundan belirle
#                 if ardisik_gun < 0:
#                     islem_yonu = 'SATIŞ'
#                     ardisik_gun_abs = abs(ardisik_gun)
#                 elif ardisik_gun > 0:
#                     islem_yonu = 'ALIŞ'
#                     ardisik_gun_abs = ardisik_gun
#                 else:
#                     islem_yonu = 'NÖTR'
#                     ardisik_gun_abs = 0

#                 stock_data = {
#                     'Broker': broker_name,
#                     'Hisse': row['Hisse'],
#                     'ArdışıkToplam': float(ardisik_toplam),
#                     'ArdışıkGün': int(ardisik_gun),  # Orijinal değeri sakla (+ veya -)
#                     'ArdışıkGünMutlak': ardisik_gun_abs,  # Mutlak değeri
#                     'DolaşımOranı': float(dolasim_orani),
#                     'ToplamGetiri': float(toplam_getiri),
#                     'İşlemYönü': islem_yonu,
#                     'DailyPositions': daily_positions,
#                     'DateColumns': date_cols
#                 }

#                 # Gelişmiş özellik mühendisliği
#                 stock_data.update(self._calculate_advanced_features(daily_positions, ardisik_toplam, ardisik_gun_abs))

#                 consolidated_list.append(stock_data)

#         self.consolidated_data = pd.DataFrame(consolidated_list)
#         print(f"Toplam {len(self.consolidated_data)} kayıt hazırlandı")
#         print(f"ALIŞ pozisyonu: {len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'ALIŞ'])}")
#         print(f"SATIŞ pozisyonu: {len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'SATIŞ'])}")
#         print(f"NÖTR pozisyon: {len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'NÖTR'])}")

#     def _calculate_advanced_features(self, daily_positions, ardisik_toplam, ardisik_gun_abs):
#         """Gelişmiş özellikler hesapla"""
#         positions = np.array(daily_positions)

#         # Temel istatistikler
#         total_volume = np.sum(np.abs(positions))
#         net_position = np.sum(positions)
#         volatility = np.std(positions) if len(positions) > 1 else 0

#         # Pozisyon analizi
#         positive_days = np.sum(positions > 0)
#         negative_days = np.sum(positions < 0)
#         zero_days = np.sum(positions == 0)

#         # Ardışık işlem gücü
#         consecutive_strength = abs(ardisik_toplam) / max(1, ardisik_gun_abs) if ardisik_gun_abs > 0 else 0

#         # Trend analizi
#         if len(positions) >= 2:
#             x = np.arange(len(positions))
#             slope, _, correlation, _, _ = stats.linregress(x, positions)
#             recent_momentum = np.mean(positions[-3:]) if len(positions) >= 3 else positions[-1] if len(positions) > 0 else 0
#             momentum_adjusted = recent_momentum / max(volatility, 0.001)
#         else:
#             slope = 0
#             correlation = 0
#             recent_momentum = 0
#             momentum_adjusted = 0

#         return {
#             'TotalVolume': total_volume,
#             'NetPosition': net_position,
#             'Volatility': volatility,
#             'PositiveDays': positive_days,
#             'NegativeDays': negative_days,
#             'ZeroDays': zero_days,
#             'ConsecutiveStrength': consecutive_strength,
#             'TrendSlope': slope,
#             'TrendCorrelation': correlation,
#             'RecentMomentum': recent_momentum,
#             'MomentumAdjusted': momentum_adjusted,
#             'MaxSingleDayVolume': np.max(np.abs(positions)) if len(positions) > 0 else 0,
#         }

#     def broker_strategy_analysis(self):
#         """Aracı kurum stratejilerini analiz et"""
#         print("\n=== ARACI KURUM STRATEJİ ANALİZİ ===")

#         broker_summary = self.consolidated_data.groupby('Broker').agg({
#             'ArdışıkToplam': ['sum', 'mean', 'count'],
#             'ArdışıkGünMutlak': ['mean', 'max', 'std'],
#             'TotalVolume': 'sum',
#             'NetPosition': 'sum',
#             'ConsecutiveStrength': 'mean',
#             'ToplamGetiri': 'mean',
#             'DolaşımOranı': 'mean'
#         }).round(3)

#         broker_summary.columns = ['_'.join(col) for col in broker_summary.columns]

#         # Strateji belirleme
#         def determine_strategy(row):
#             net_pos = row['NetPosition_sum']
#             avg_consecutive = row['ArdışıkGünMutlak_mean']
#             total_ops = row['ArdışıkToplam_count']

#             if net_pos > 0 and avg_consecutive > 3:
#                 return 'Aggressive_Buyer'
#             elif net_pos < 0 and avg_consecutive > 3:
#                 return 'Aggressive_Seller'
#             elif avg_consecutive > 5:
#                 return 'Long_Term_Player'
#             elif total_ops > 20:
#                 return 'Active_Trader'
#             else:
#                 return 'Selective_Trader'

#         broker_summary['Strategy_Type'] = broker_summary.apply(determine_strategy, axis=1)

#         print("Aracı Kurum Strateji Dağılımı:")
#         print(broker_summary['Strategy_Type'].value_counts())
#         print("\nDetaylı Analiz:")
#         for strategy in broker_summary['Strategy_Type'].unique():
#             brokers = broker_summary[broker_summary['Strategy_Type'] == strategy].index.tolist()
#             print(f"{strategy}: {brokers}")

#         return broker_summary

#     def same_direction_consensus_analysis(self):
#         """Aynı hissede aynı yönde hareket eden kurumları analiz et"""
#         print("\n=== KONSENSÜS VE KARŞIT GÖRÜŞ ANALİZİ ===")

#         consensus_data = []

#         # Hisse bazlı grup analizi
#         for stock in self.consolidated_data['Hisse'].unique():
#             stock_data = self.consolidated_data[self.consolidated_data['Hisse'] == stock].copy()

#             if len(stock_data) < 2:
#                 continue

#             # Alış ve satış yapan kurumlar
#             buyers = stock_data[stock_data['İşlemYönü'] == 'ALIŞ']
#             sellers = stock_data[stock_data['İşlemYönü'] == 'SATIŞ']

#             buy_volume = buyers['ArdışıkToplam'].sum() if len(buyers) > 0 else 0
#             sell_volume = abs(sellers['ArdışıkToplam'].sum()) if len(sellers) > 0 else 0

#             # Konsensüs skoru
#             total_brokers = len(stock_data)
#             buy_brokers = len(buyers)
#             sell_brokers = len(sellers)

#             if total_brokers > 0:
#                 buy_consensus = buy_brokers / total_brokers
#                 sell_consensus = sell_brokers / total_brokers
#                 dominant_consensus = max(buy_consensus, sell_consensus)
#             else:
#                 buy_consensus = sell_consensus = dominant_consensus = 0

#             # Net akış
#             net_flow = buy_volume - sell_volume

#             # Ortalama getiri
#             avg_return = stock_data['ToplamGetiri'].mean()

#             consensus_data.append({
#                 'Hisse': stock,
#                 'TotalBrokers': total_brokers,
#                 'BuyBrokers': buy_brokers,
#                 'SellBrokers': sell_brokers,
#                 'BuyVolume': buy_volume,
#                 'SellVolume': sell_volume,
#                 'NetFlow': net_flow,
#                 'BuyConsensus': buy_consensus,
#                 'SellConsensus': sell_consensus,
#                 'DominantConsensus': dominant_consensus,
#                 'AvgReturn': avg_return,
#                 'ConsensusScore': dominant_consensus * abs(net_flow),
#                 'BuyerList': buyers['Broker'].tolist(),
#                 'SellerList': sellers['Broker'].tolist()
#             })

#         consensus_df = pd.DataFrame(consensus_data)

#         # En güçlü konsensüs (aynı yönde)
#         strong_consensus = consensus_df[consensus_df['DominantConsensus'] >= 0.7].sort_values('ConsensusScore', ascending=False)

#         print("GÜÇLÜ KONSENSÜS (Brokerların %70+ aynı yönde):")
#         print("="*50)
#         for _, row in strong_consensus.head(10).iterrows():
#             direction = "ALIŞ" if row['BuyConsensus'] > row['SellConsensus'] else "SATIŞ"
#             dominant_list = row['BuyerList'] if direction == "ALIŞ" else row['SellerList']
#             minority_list = row['SellerList'] if direction == "ALIŞ" else row['BuyerList']

#             print(f"{row['Hisse']}: {direction} yönünde %{row['DominantConsensus']*100:.0f} konsensüs")
#             print(f"  Dominant: {dominant_list}")
#             if minority_list:
#                 print(f"  Karşıt: {minority_list}")
#             print(f"  Net Akış: {row['NetFlow']:,.0f} lot, Getiri: %{row['AvgReturn']:.2f}")
#             print()

#         # Bölünmüş görüş (50-50 civarı)
#         divided_opinion = consensus_df[
#             (consensus_df['DominantConsensus'] < 0.65) &
#             (consensus_df['TotalBrokers'] >= 4)
#         ].sort_values('TotalBrokers', ascending=False)

#         print("BÖLÜNMÜŞ GÖRÜŞ (Belirsiz konsensüs):")
#         print("="*40)
#         for _, row in divided_opinion.head(5).iterrows():
#             print(f"{row['Hisse']}: {row['BuyBrokers']} Alıcı vs {row['SellBrokers']} Satıcı")
#             print(f"  Alıcılar: {row['BuyerList']}")
#             print(f"  Satıcılar: {row['SellerList']}")
#             print(f"  Net Akış: {row['NetFlow']:,.0f} lot, Getiri: %{row['AvgReturn']:.2f}")
#             print()

#         return consensus_df

#     def broker_correlation_analysis(self):
#         """Brokerların birbirleriyle korelasyon analizi"""
#         print("\n=== BROKER KORELASYON ANALİZİ ===")

#         # Broker-Hisse matrix oluştur
#         pivot_data = self.consolidated_data.pivot_table(
#             index='Hisse',
#             columns='Broker',
#             values='ArdışıkToplam',
#             fill_value=0
#         )

#         if pivot_data.shape[1] < 2:
#             print("Korelasyon analizi için yeterli broker yok")
#             return None

#         # Korelasyon matrisi
#         broker_corr = pivot_data.corr()

#         # En yüksek korelasyonları bul
#         correlations = []
#         for i in range(len(broker_corr.columns)):
#             for j in range(i+1, len(broker_corr.columns)):
#                 broker1 = broker_corr.columns[i]
#                 broker2 = broker_corr.columns[j]
#                 corr_value = broker_corr.iloc[i, j]

#                 if not pd.isna(corr_value):
#                     correlations.append({
#                         'Kurum1': broker1,
#                         'Kurum2': broker2,
#                         'Korelasyon': corr_value
#                     })

#         corr_df = pd.DataFrame(correlations).sort_values('Korelasyon', ascending=False)

#         print("EN YÜKSEK POZİTİF KORELASYON (Aynı yöndehareket):")
#         print(corr_df.head(10))

#         print("\nEN YÜKSEK NEGATİF KORELASYON (Ters yönde hareket):")
#         print(corr_df.tail(10))

#         # Cluster analizi
#         if len(broker_corr) > 2:
#             # Korelasyon distance matrix
#             distance_matrix = 1 - broker_corr.abs()

#             print(f"\nKurum Benzerlik Analizi:")
#             print("(1'e yakın = çok benzer, 0'a yakın = çok farklı)")
#             similarity_scores = broker_corr.abs().mean().sort_values(ascending=False)
#             print(similarity_scores.head(10))

#         return corr_df, broker_corr

#     def smart_money_analysis(self):
#         """Akıllı para hareketlerini tespit et"""
#         print("\n=== SMART MONEY ANALİZİ ===")

#         stock_analysis = []

#         for stock in self.consolidated_data['Hisse'].unique():
#             stock_data = self.consolidated_data[self.consolidated_data['Hisse'] == stock].copy()

#             if len(stock_data) < 2:
#                 continue

#             # Net pozisyon toplamları
#             total_buy_volume = stock_data[stock_data['İşlemYönü'] == 'ALIŞ']['ArdışıkToplam'].sum()
#             total_sell_volume = abs(stock_data[stock_data['İşlemYönü'] == 'SATIŞ']['ArdışıkToplam'].sum())
#             net_flow = total_buy_volume - total_sell_volume

#             # Getiri korelasyonu
#             avg_return = stock_data['ToplamGetiri'].mean()

#             # Consensus analizi
#             buy_brokers = len(stock_data[stock_data['İşlemYönü'] == 'ALIŞ'])
#             sell_brokers = len(stock_data[stock_data['İşlemYönü'] == 'SATIŞ'])
#             total_brokers = len(stock_data)

#             consensus_score = abs(buy_brokers - sell_brokers) / total_brokers if total_brokers > 0 else 0

#             # Smart money skoru
#             total_volume = total_buy_volume + total_sell_volume
#             if total_volume > 0:
#                 smart_score = (net_flow * avg_return * consensus_score) / total_volume
#             else:
#                 smart_score = 0

#             stock_analysis.append({
#                 'Hisse': stock,
#                 'NetFlow': net_flow,
#                 'AvgReturn': avg_return,
#                 'ConsensusScore': consensus_score,
#                 'SmartScore': smart_score,
#                 'BuyBrokers': buy_brokers,
#                 'SellBrokers': sell_brokers,
#                 'TotalVolume': total_volume
#             })

#         smart_df = pd.DataFrame(stock_analysis).sort_values('SmartScore', ascending=False)

#         print("En Yüksek Smart Money Skorları:")
#         print(smart_df.head(10)[['Hisse', 'SmartScore', 'NetFlow', 'AvgReturn', 'ConsensusScore']])

#         print("\nEn Düşük Smart Money Skorları:")
#         print(smart_df.tail(10)[['Hisse', 'SmartScore', 'NetFlow', 'AvgReturn', 'ConsensusScore']])

#         return smart_df

#     def consecutive_success_analysis(self):
#         """Ardışık işlemlerin başarı oranlarını analiz et"""
#         print("\n=== ARDIŞIK İŞLEM BAŞARI ANALİZİ ===")

#         # Başarı kriteri: Ardışık işlem yönü ile getiri yönü uyumlu mu?
#         def calculate_success(row):
#             if row['İşlemYönü'] == 'ALIŞ':
#                 return 1 if row['ToplamGetiri'] > 0 else 0
#             elif row['İşlemYönü'] == 'SATIŞ':
#                 return 1 if row['ToplamGetiri'] < 0 else 0
#             else:
#                 return 0

#         self.consolidated_data['Success'] = self.consolidated_data.apply(calculate_success, axis=1)

#         # ArdışıkGün sayısına göre başarı analizi
#         success_by_days = self.consolidated_data.groupby('ArdışıkGünMutlak').agg({
#             'Success': ['sum', 'count', 'mean'],
#             'ArdışıkToplam': ['mean', 'std'],
#             'ToplamGetiri': ['mean', 'std']
#         }).round(3)

#         success_by_days.columns = ['_'.join(col) for col in success_by_days.columns]
#         success_by_days['SuccessRate'] = success_by_days['Success_mean'] * 100

#         print("ArdışıkGün Sayısına Göre Başarı Oranları:")
#         valid_days = success_by_days[success_by_days['Success_count'] >= 3]  # En az 3 örnek
#         print(valid_days[['Success_count', 'SuccessRate', 'ToplamGetiri_mean']])

#         # Broker bazlı başarı analizi
#         broker_success = self.consolidated_data.groupby(['Broker', 'İşlemYönü']).agg({
#             'Success': ['sum', 'count', 'mean'],
#             'ArdışıkToplam': 'mean',
#             'ToplamGetiri': 'mean'
#         }).round(3)

#         broker_success.columns = ['_'.join(col) for col in broker_success.columns]
#         broker_success['SuccessRate'] = broker_success['Success_mean'] * 100

#         print("\nBroker ve Yön Bazlı Başarı Oranları (En Az 3 İşlem):")
#         valid_broker_success = broker_success[broker_success['Success_count'] >= 3].sort_values('SuccessRate', ascending=False)
#         print(valid_broker_success[['Success_count', 'SuccessRate', 'ToplamGetiri_mean']].head(15))

#         return success_by_days, broker_success

#     def anomaly_detection(self):
#         """Anormal davranış gösteren işlemleri tespit et"""
#         print("\n=== ANOMALİ TESPİT ANALİZİ ===")

#         # Özellik seçimi
#         feature_cols = ['ArdışıkToplam', 'ArdışıkGünMutlak', 'TotalVolume', 'Volatility',
#                        'ConsecutiveStrength', 'MaxSingleDayVolume', 'DolaşımOranı']

#         features = self.consolidated_data[feature_cols].fillna(0)

#         # Isolation Forest ile anomali tespiti
#         iso_forest = IsolationForest(contamination=0.05, random_state=42)
#         anomaly_labels = iso_forest.fit_predict(features)

#         self.consolidated_data['Anomaly'] = anomaly_labels

#         anomalies = self.consolidated_data[self.consolidated_data['Anomaly'] == -1]

#         print(f"Tespit edilen anomali sayısı: {len(anomalies)}")

#         if len(anomalies) > 0:
#             print("\nEn büyük anomaliler:")
#             anomaly_display = anomalies.nlargest(10, 'ArdışıkToplam')[
#                 ['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkToplam', 'ArdışıkGünMutlak', 'ToplamGetiri']
#             ]
#             print(anomaly_display)

#             print("\nAnomali Türü Analizi:")
#             print("Yüksek Hacim Anomalileri:", len(anomalies[abs(anomalies['ArdışıkToplam']) > anomalies['ArdışıkToplam'].quantile(0.95)]))
#             print("Uzun Süre Anomalileri:", len(anomalies[anomalies['ArdışıkGünMutlak'] > anomalies['ArdışıkGünMutlak'].quantile(0.95)]))

#         return anomalies

#     def momentum_timing_analysis(self):
#         """Momentum ve timing analizi"""
#         print("\n=== MOMENTUM VE TİMİNG ANALİZİ ===")

#         # Güçlü momentum hareketlerini tespit et
#         volume_threshold = self.consolidated_data['ArdışıkToplam'].quantile(0.75)
#         strong_momentum = self.consolidated_data[
#             (self.consolidated_data['ArdışıkGünMutlak'] >= 3) &
#             (abs(self.consolidated_data['ArdışıkToplam']) > volume_threshold)
#         ].copy()

#         print(f"Güçlü momentum hareketi sayısı: {len(strong_momentum)}")

#         if len(strong_momentum) > 0:
#             print("\nEn Güçlü Momentum Hareketleri:")
#             momentum_display = strong_momentum.nlargest(10, 'ConsecutiveStrength')[
#                 ['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkGünMutlak', 'ArdışıkToplam', 'ToplamGetiri', 'ConsecutiveStrength']
#             ]
#             print(momentum_display)

#             # Momentum başarı analizi
#             momentum_success = strong_momentum.groupby('İşlemYönü').agg({
#                 'Success': 'mean',
#                 'ToplamGetiri': 'mean',
#                 'ArdışıkGünMutlak': 'mean',
#                 'ConsecutiveStrength': 'mean'
#             }).round(3)

#             print("\nMomentum Türü Başarı Analizi:")
#             print(momentum_success)

#         return strong_momentum

#     def sector_analysis(self):
#         """Sektör bazlı analiz"""
#         print("\n=== SEKTÖR ANALİZİ ===")

#         SECTOR_STOCKS = {
#             'Banka':         ['AKBNK', 'ISCTR', 'YKBNK', 'GARAN', 'HALKB', 'VAKBN', 'DENIZ', 'FINBN'],
#             'Perakende':     ['BIMAS', 'SOKM', 'MIGROS', 'AYGAZ', 'ULKER'],
#             'Ulaştırma':     ['THYAO', 'PGSUS', 'CLEBI', 'TAVHL', 'ALARK'],
#             'Enerji/Sanayi': ['TUPRS', 'PETKM', 'ENJSA', 'ENERYA', 'OGEN', 'AKSEN', 'AYGAZ'],
#             'Telekom':       ['TCELL', 'TTKOM'],
#             'Holding':       ['KCHOL', 'SAHOL', 'DOHOL', 'DOGHL'],
#             'Savunma/Teknoloji': ['ASELS', 'OTKAR', 'PENTA', 'REEDR'],
#             'Çimento/İnşaat': ['EREGL', 'CIMSA', 'KONYA', 'AVRASYA', 'ENKAI', 'AKCNS'],
#             'Gıda/İçecek':   ['CCOLA', 'ULKER', 'EFES'],
#             'Sigorta/Finansal Hizmetler': ['AVIVA', 'HEKTS'],
#             'GYO':           ['AVGYO', 'ATAGY', 'ISGYO']
#         }

#         def categorize_stock(stock_code):
#             code = stock_code.replace('.E', '').upper().strip()
#             for sector, stocks in SECTOR_STOCKS.items():
#                 if code in stocks:
#                     return sector
#             return 'Diğer'

#         self.consolidated_data['Sector'] = self.consolidated_data['Hisse'].apply(categorize_stock)

#         # Sektör bazlı özet
#         sector_summary = self.consolidated_data.groupby(['Sector', 'İşlemYönü']).agg({
#             'ArdışıkToplam': ['sum', 'mean', 'count'],
#             'ToplamGetiri': ['mean', 'std'],
#             'ArdışıkGünMutlak': 'mean',
#             'Success': 'mean'
#         }).round(3)

#         sector_summary.columns = ['_'.join(col) for col in sector_summary.columns]

#         print("Sektör Bazlı İşlem Özeti:")
#         for sector in self.consolidated_data['Sector'].unique():
#             sector_data = sector_summary.loc[sector] if sector in sector_summary.index else None
#             if sector_data is not None:
#                 print(f"\n{sector}:")
#                 print(sector_data)

#         # Sektör net akışları
#         sector_flows = self.consolidated_data.groupby('Sector').agg({
#             'ArdışıkToplam': 'sum',
#             'ToplamGetiri': 'mean',
#             'Success': 'mean'
#         }).sort_values('ArdışıkToplam', ascending=False)

#         print("\nSektör Net Akış Sıralaması:")
#         print(sector_flows)

#         return sector_summary, sector_flows

#     def success_prediction_analysis(self):
#         """Başarı tahmin analizi"""
#         print("\n=== BAŞARI TAHMİN ANALİZİ ===")

#         # Özellik seçimi
#         feature_columns = [
#             'ArdışıkGünMutlak', 'TotalVolume', 'ConsecutiveStrength',
#             'TrendSlope', 'RecentMomentum', 'MomentumAdjusted',
#             'DolaşımOranı', 'MaxSingleDayVolume', 'Volatility'
#         ]

#         # Kategorik encoding
#         self.consolidated_data['Broker_encoded'] = pd.Categorical(self.consolidated_data['Broker']).codes
#         self.consolidated_data['Direction_encoded'] = self.consolidated_data['İşlemYönü'].map({'ALIŞ': 1, 'SATIŞ': -1, 'NÖTR': 0})

#         feature_columns.extend(['Broker_encoded', 'Direction_encoded'])

#         # Model eğitimi
#         X = self.consolidated_data[feature_columns].fillna(0)
#         y = self.consolidated_data['Success']

#         if len(X) < 10 or y.sum() < 3:
#             print("Model eğitimi için yeterli veri yok")
#             return None

#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

#         model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
#         model.fit(X_train, y_train)

#         # Tahmin
#         y_pred = model.predict(X_test)
#         y_pred_proba = model.predict_proba(X_test)[:, 1]

#         print("Model Performansı:")
#         print(classification_report(y_test, y_pred))

#         # Özellik önemlilikleri
#         feature_importance = pd.DataFrame({
#             'Feature': feature_columns,
#             'Importance': model.feature_importances_
#         }).sort_values('Importance', ascending=False)

#         print("\nEn Önemli Özellikler:")
#         print(feature_importance.head(8))

#         # Yüksek olasılıklı tahminler
#         test_indices = X_test.index
#         high_prob_data = self.consolidated_data.loc[test_indices].copy()
#         high_prob_data['Success_Probability'] = y_pred_proba

#         high_confidence = high_prob_data[high_prob_data['Success_Probability'] > 0.7].sort_values('Success_Probability', ascending=False)

#         if len(high_confidence) > 0:
#             print(f"\nYüksek Başarı Olasılığı (>%70): {len(high_confidence)} işlem")
#             print(high_confidence[['Broker', 'Hisse', 'İşlemYönü', 'ArdışıkToplam', 'ToplamGetiri', 'Success_Probability']].head())

#         return model, feature_importance, high_confidence

#     # ========== YENİ GELİŞMİŞ ANALİZ FONKSİYONLARI ==========

#     def herd_behavior_analysis(self):
#         """Sürü davranışı tespiti - Brokerların aynı anda aynı hareketi yapması"""
#         print("\n=== SÜRÜ DAVRANIŞI ANALİZİ ===")

#         # Hisse bazlı broker hareketlerini grupla
#         herd_data = []

#         for stock in self.consolidated_data['Hisse'].unique():
#             stock_data = self.consolidated_data[self.consolidated_data['Hisse'] == stock].copy()

#             if len(stock_data) < 2:
#                 continue

#             # Aynı yönde hareket eden broker sayısı
#             buy_brokers = len(stock_data[stock_data['İşlemYönü'] == 'ALIŞ'])
#             sell_brokers = len(stock_data[stock_data['İşlemYönü'] == 'SATIŞ'])
#             total_brokers = len(stock_data)

#             # Sürü davranışı skoru (dominant yönün yüzdesi)
#             if total_brokers > 0:
#                 herd_score = max(buy_brokers, sell_brokers) / total_brokers
#                 dominant_direction = 'ALIŞ' if buy_brokers > sell_brokers else 'SATIŞ'
#                 minority_brokers = min(buy_brokers, sell_brokers)
#             else:
#                 herd_score = 0
#                 dominant_direction = 'NÖTR'
#                 minority_brokers = 0

#             # Hacim ağırlıklı sürü skoru
#             total_volume = abs(stock_data['ArdışıkToplam']).sum()
#             dominant_volume = abs(stock_data[stock_data['İşlemYönü'] == dominant_direction]['ArdışıkToplam']).sum()
#             volume_herd_score = dominant_volume / total_volume if total_volume > 0 else 0

#             # Sürü gücü (broker sayısı ve hacim kombinasyonu)
#             herd_strength = (herd_score * 0.6) + (volume_herd_score * 0.4)

#             herd_data.append({
#                 'Hisse': stock,
#                 'TotalBrokers': total_brokers,
#                 'DominantDirection': dominant_direction,
#                 'HerdScore': herd_score,
#                 'VolumeHerdScore': volume_herd_score,
#                 'HerdStrength': herd_strength,
#                 'MinorityBrokers': minority_brokers,
#                 'TotalVolume': total_volume,
#                 'AvgReturn': stock_data['ToplamGetiri'].mean()
#             })

#         herd_df = pd.DataFrame(herd_data).sort_values('HerdStrength', ascending=False)

#         # Güçlü sürü davranışı (>%80 konsensüs)
#         strong_herd = herd_df[herd_df['HerdScore'] >= 0.8]
#         print(f"Güçlü sürü davranışı tespit edilen hisse sayısı: {len(strong_herd)}")

#         if len(strong_herd) > 0:
#             print("\nEn Güçlü Sürü Davranışları:")
#             print(strong_herd.head(10)[['Hisse', 'DominantDirection', 'HerdScore', 'TotalBrokers', 'HerdStrength', 'AvgReturn']])

#         # Bölünmüş davranış (yaklaşık 50-50)
#         divided_behavior = herd_df[(herd_df['HerdScore'] < 0.6) & (herd_df['TotalBrokers'] >= 4)]
#         print(f"\nBölünmüş davranış gösteren hisse sayısı: {len(divided_behavior)}")

#         if len(divided_behavior) > 0:
#             print("En Bölünmüş Davranışlar:")
#             print(divided_behavior.head(5)[['Hisse', 'TotalBrokers', 'HerdScore', 'MinorityBrokers', 'AvgReturn']])

#         return herd_df

#     def weekend_effect_analysis(self):
#         """Hafta içi günlerde alım/satım pattern analizi"""
#         print("\n=== HAFTA İÇİ PATTERN ANALİZİ ===")

#         # Tarih sütunlarından günlük pozisyonları çıkar
#         daily_patterns = []

#         for _, row in self.consolidated_data.iterrows():
#             if 'DateColumns' in row and 'DailyPositions' in row:
#                 date_columns = row['DateColumns']
#                 daily_positions = row['DailyPositions']

#                 for i, (date_str, position) in enumerate(zip(date_columns, daily_positions)):
#                     try:
#                         date_obj = pd.to_datetime(date_str, format='%d/%m/%Y')
#                         weekday = date_obj.dayofweek  # 0=Pazartesi, 4=Cuma
#                         weekday_name = date_obj.strftime('%A')

#                         daily_patterns.append({
#                             'Broker': row['Broker'],
#                             'Hisse': row['Hisse'],
#                             'Tarih': date_obj,
#                             'Weekday': weekday,
#                             'WeekdayName': weekday_name,
#                             'Position': float(position),
#                             'IsPositive': position > 0,
#                             'AbsPosition': abs(float(position))
#                         })
#                     except:
#                         continue

#         if not daily_patterns:
#             print("Günlük pozisyon verisi bulunamadı")
#             return None

#         pattern_df = pd.DataFrame(daily_patterns)

#         # Gün bazlı toplam aktivite
#         daily_summary = pattern_df.groupby('WeekdayName').agg({
#             'Position': ['sum', 'mean', 'count'],
#             'AbsPosition': ['sum', 'mean'],
#             'IsPositive': 'mean'
#         }).round(3)

#         daily_summary.columns = ['_'.join(col) for col in daily_summary.columns]
#         daily_summary['NetFlow'] = daily_summary['Position_sum']
#         daily_summary['AvgDailyVolume'] = daily_summary['AbsPosition_sum']
#         daily_summary['BuyRatio'] = daily_summary['IsPositive_mean']

#         print("Günlere Göre İşlem Aktivitesi:")
#         print(daily_summary[['Position_count', 'NetFlow', 'AvgDailyVolume', 'BuyRatio']])

#         # Broker bazlı gün tercihleri
#         broker_day_pref = pattern_df.groupby(['Broker', 'WeekdayName']).agg({
#             'AbsPosition': 'sum',
#             'Position': 'sum'
#         }).unstack(fill_value=0)

#         # En aktif gün/broker kombinasyonları
#         broker_activity = pattern_df.groupby(['Broker', 'WeekdayName']).size().unstack(fill_value=0)

#         print("\nBroker Günlük Aktivite Tercihleri (İşlem Sayısı):")
#         for broker in broker_activity.index[:5]:  # İlk 5 broker
#             max_day = broker_activity.loc[broker].idxmax()
#             max_count = broker_activity.loc[broker].max()
#             print(f"{broker}: En aktif {max_day} ({max_count} işlem)")

#         return pattern_df, daily_summary, broker_activity

#     def broker_network_analysis(self):
#         """Hisse bazında broker ilişki ağları"""
#         print("\n=== BROKER İLİŞKİ AĞI ANALİZİ ===")

#         # Hisse bazlı broker network
#         stock_networks = {}

#         for stock in self.consolidated_data['Hisse'].unique():
#             stock_data = self.consolidated_data[self.consolidated_data['Hisse'] == stock]

#             if len(stock_data) < 2:
#                 continue

#             # Bu hissede işlem yapan brokerlar
#             brokers = stock_data['Broker'].tolist()

#             # Network graph oluştur
#             G = nx.Graph()

#             # Brokerları node olarak ekle
#             for broker in brokers:
#                 broker_info = stock_data[stock_data['Broker'] == broker].iloc[0]
#                 G.add_node(broker,
#                           direction=broker_info['İşlemYönü'],
#                           volume=abs(broker_info['ArdışıkToplam']),
#                           return_rate=broker_info['ToplamGetiri'])

#             # Aynı yönde hareket eden brokerlar arasında bağlantı oluştur
#             for i, broker1 in enumerate(brokers):
#                 for j, broker2 in enumerate(brokers[i+1:], i+1):
#                     broker1_data = stock_data[stock_data['Broker'] == broker1].iloc[0]
#                     broker2_data = stock_data[stock_data['Broker'] == broker2].iloc[0]

#                     # Aynı yönde hareket ediyorlarsa bağlantı kur
#                     if broker1_data['İşlemYönü'] == broker2_data['İşlemYönü'] != 'NÖTR':
#                         # Bağlantı ağırlığı: hacim benzerliği
#                         vol1 = abs(broker1_data['ArdışıkToplam'])
#                         vol2 = abs(broker2_data['ArdışıkToplam'])
#                         similarity = 1 - abs(vol1 - vol2) / max(vol1 + vol2, 1)

#                         G.add_edge(broker1, broker2, weight=similarity)

#             if len(G.nodes) >= 2:
#                 stock_networks[stock] = G

#         # Network metrics hesapla
#         network_metrics = []

#         for stock, G in stock_networks.items():
#             if len(G.nodes) == 0:
#                 continue

#             # Temel network metrikleri
#             density = nx.density(G)
#             num_components = nx.number_connected_components(G)

#             # En merkezi broker (degree centrality)
#             centrality = nx.degree_centrality(G)
#             most_central = max(centrality, key=centrality.get) if centrality else None

#             # Clustering coefficient
#             clustering = nx.average_clustering(G)

#             network_metrics.append({
#                 'Hisse': stock,
#                 'TotalBrokers': len(G.nodes),
#                 'TotalConnections': len(G.edges),
#                 'Density': density,
#                 'Components': num_components,
#                 'MostCentral': most_central,
#                 'CentralityScore': centrality.get(most_central, 0) if most_central else 0,
#                 'Clustering': clustering
#             })

#         network_df = pd.DataFrame(network_metrics).sort_values('Density', ascending=False)

#         print("En Yoğun Broker Ağları (Yüksek İşbirliği):")
#         print(network_df.head(10)[['Hisse', 'TotalBrokers', 'Density', 'MostCentral', 'CentralityScore']])

#         return stock_networks, network_df

#     def central_node_analysis(self):
#         """En etkili broker ve hisselerin tespiti"""
#         print("\n=== MERKEZ NODE ANALİZİ ===")

#         # Global broker network oluştur
#         G_global = nx.Graph()

#         # Tüm brokerları node olarak ekle
#         broker_stats = self.consolidated_data.groupby('Broker').agg({
#             'ArdışıkToplam': 'sum',
#             'Hisse': 'count',
#             'ToplamGetiri': 'mean',
#             'Success': 'mean'
#         }).round(3)

#         for broker, stats in broker_stats.iterrows():
#             G_global.add_node(broker,
#                             total_volume=stats['ArdışıkToplam'],
#                             stock_count=stats['Hisse'],
#                             avg_return=stats['ToplamGetiri'],
#                             success_rate=stats['Success'])

#         # Ortak hisselerde işlem yapan brokerlar arasında bağlantı
#         broker_stocks = self.consolidated_data.groupby('Broker')['Hisse'].apply(set).to_dict()

#         for broker1 in broker_stocks:
#             for broker2 in broker_stocks:
#                 if broker1 < broker2:  # Çift sayımı önle
#                     common_stocks = broker_stocks[broker1] & broker_stocks[broker2]
#                     if len(common_stocks) > 0:
#                         # Bağlantı ağırlığı: ortak hisse sayısı
#                         weight = len(common_stocks)
#                         G_global.add_edge(broker1, broker2, weight=weight)

#         # Centrality metrikleri
#         degree_centrality = nx.degree_centrality(G_global)
#         betweenness_centrality = nx.betweenness_centrality(G_global, weight='weight')
#         closeness_centrality = nx.closeness_centrality(G_global)
#         eigenvector_centrality = nx.eigenvector_centrality(G_global, weight='weight', max_iter=1000)

#         # Merkezi broker analizi
#         central_analysis = []
#         for broker in G_global.nodes():
#             central_analysis.append({
#                 'Broker': broker,
#                 'DegreeCentrality': degree_centrality.get(broker, 0),
#                 'BetweennessCentrality': betweenness_centrality.get(broker, 0),
#                 'ClosenessCentrality': closeness_centrality.get(broker, 0),
#                 'EigenvectorCentrality': eigenvector_centrality.get(broker, 0),
#                 'TotalVolume': G_global.nodes[broker]['total_volume'],
#                 'StockCount': G_global.nodes[broker]['stock_count'],
#                 'AvgReturn': G_global.nodes[broker]['avg_return']
#             })

#         central_df = pd.DataFrame(central_analysis)

#         # Kombine merkezi skor
#         central_df['CombinedCentrality'] = (
#             central_df['DegreeCentrality'] * 0.3 +
#             central_df['BetweennessCentrality'] * 0.3 +
#             central_df['EigenvectorCentrality'] * 0.4
#         )

#         central_df = central_df.sort_values('CombinedCentrality', ascending=False)

#         print("En Merkezi Brokerlar (Ağdaki Önem Sırasına Göre):")
#         print(central_df.head(10)[['Broker', 'CombinedCentrality', 'StockCount', 'TotalVolume', 'AvgReturn']])

#         # Hisse bazlı merkezi analiz
#         stock_centrality = []
#         stock_broker_count = self.consolidated_data.groupby('Hisse').size()
#         stock_volume = self.consolidated_data.groupby('Hisse')['ArdışıkToplam'].apply(lambda x: abs(x).sum())
#         stock_return = self.consolidated_data.groupby('Hisse')['ToplamGetiri'].mean()

#         for stock in self.consolidated_data['Hisse'].unique():
#             centrality_score = (
#                 stock_broker_count.get(stock, 0) * 0.4 +  # Broker ilgisi
#                 (stock_volume.get(stock, 0) / 1000) * 0.4 +  # Hacim
#                 abs(stock_return.get(stock, 0)) * 0.2  # Getiri
#             )

#             stock_centrality.append({
#                 'Hisse': stock,
#                 'BrokerCount': stock_broker_count.get(stock, 0),
#                 'TotalVolume': stock_volume.get(stock, 0),
#                 'AvgReturn': stock_return.get(stock, 0),
#                 'CentralityScore': centrality_score
#             })

#         stock_central_df = pd.DataFrame(stock_centrality).sort_values('CentralityScore', ascending=False)

#         print("\nEn Merkezi Hisseler (Broker İlgisi ve Hacim Bazında):")
#         print(stock_central_df.head(10)[['Hisse', 'BrokerCount', 'TotalVolume', 'AvgReturn', 'CentralityScore']])

#         return central_df, stock_central_df, G_global

#     def community_detection_analysis(self):
#         """Doğal broker gruplarının keşfi"""
#         print("\n=== TOPLULUK TESPİT ANALİZİ ===")

#         # Broker benzerlik matrisi oluştur
#         broker_pivot = self.consolidated_data.pivot_table(
#             index='Hisse',
#             columns='Broker',
#             values='ArdışıkToplam',
#             fill_value=0
#         )

#         if broker_pivot.shape[1] < 3:
#             print("Topluluk analizi için yeterli broker yok")
#             return None

#         # Korelasyon bazlı benzerlik
#         broker_corr = broker_pivot.corr().fillna(0)

#         # Network graph oluştur (korelasyon > 0.3 olan brokerlar arası bağlantı)
#         G_community = nx.Graph()

#         for broker in broker_corr.columns:
#             # Node attributes
#             broker_data = self.consolidated_data[self.consolidated_data['Broker'] == broker]
#             G_community.add_node(broker,
#                                total_volume=abs(broker_data['ArdışıkToplam']).sum(),
#                                stock_count=len(broker_data),
#                                avg_return=broker_data['ToplamGetiri'].mean(),
#                                success_rate=broker_data['Success'].mean())

#         # Korelasyon eşiği üzerindeki brokerlar arasında bağlantı
#         correlation_threshold = 0.3
#         for i, broker1 in enumerate(broker_corr.columns):
#             for j, broker2 in enumerate(broker_corr.columns[i+1:], i+1):
#                 corr_value = broker_corr.iloc[i, j]
#                 if abs(corr_value) > correlation_threshold:
#                     G_community.add_edge(broker1, broker2, weight=abs(corr_value))

#         # Community detection algoritmaları
#         communities_dict = {}

#         try:
#             # Greedy modularity communities
#             communities_greedy = list(nx.community.greedy_modularity_communities(G_community, weight='weight'))
#             communities_dict['Greedy'] = communities_greedy

#             print(f"Greedy Modularity ile {len(communities_greedy)} topluluk tespit edildi:")
#             for i, community in enumerate(communities_greedy):
#                 community_list = list(community)
#                 if len(community_list) > 1:
#                     print(f"  Grup {i+1}: {community_list}")
#         except:
#             print("Greedy modularity analizi başarısız")

#         try:
#             # Louvain communities (eğer mevcut ise)
#             if hasattr(nx.community, 'louvain_communities'):
#                 communities_louvain = list(nx.community.louvain_communities(G_community, weight='weight'))
#                 communities_dict['Louvain'] = communities_louvain

#                 print(f"\nLouvain algoritması ile {len(communities_louvain)} topluluk tespit edildi:")
#                 for i, community in enumerate(communities_louvain):
#                     community_list = list(community)
#                     if len(community_list) > 1:
#                         print(f"  Grup {i+1}: {community_list}")
#         except:
#             pass

#         # Hierarchical clustering ile de gruplandırma yap
#         try:
#             # Distance matrix (1 - correlation)
#             distance_matrix = 1 - broker_corr.abs()

#             # Hierarchical clustering
#             n_clusters = min(5, len(broker_corr.columns) // 2)  # Makul cluster sayısı
#             clustering = AgglomerativeClustering(n_clusters=n_clusters, metric='precomputed', linkage='complete')

#             cluster_labels = clustering.fit_predict(distance_matrix)

#             # Cluster sonuçlarını grupla
#             hierarchical_communities = {}
#             for broker, label in zip(broker_corr.columns, cluster_labels):
#                 if label not in hierarchical_communities:
#                     hierarchical_communities[label] = []
#                 hierarchical_communities[label].append(broker)

#             print(f"\nHierarchical Clustering ile {len(hierarchical_communities)} grup:")
#             for cluster_id, brokers in hierarchical_communities.items():
#                 if len(brokers) > 1:
#                     print(f"  Küme {cluster_id}: {brokers}")

#             communities_dict['Hierarchical'] = list(hierarchical_communities.values())
#         except Exception as e:
#             print(f"Hierarchical clustering hatası: {e}")

#         # Topluluk kalitesi analizi
#         community_analysis = []

#         for method, communities in communities_dict.items():
#             for i, community in enumerate(communities):
#                 if len(community) > 1:
#                     community_brokers = list(community)

#                     # Topluluk içi ortalama korelasyon
#                     internal_corr = []
#                     for j, broker1 in enumerate(community_brokers):
#                         for k, broker2 in enumerate(community_brokers[j+1:], j+1):
#                             if broker1 in broker_corr.columns and broker2 in broker_corr.columns:
#                                 internal_corr.append(broker_corr.loc[broker1, broker2])

#                     avg_internal_corr = np.mean(internal_corr) if internal_corr else 0

#                     # Topluluk özellikleri
#                     community_data = self.consolidated_data[self.consolidated_data['Broker'].isin(community_brokers)]

#                     community_analysis.append({
#                         'Method': method,
#                         'CommunityID': i,
#                         'Brokers': community_brokers,
#                         'Size': len(community_brokers),
#                         'InternalCorrelation': avg_internal_corr,
#                         'TotalVolume': abs(community_data['ArdışıkToplam']).sum(),
#                         'AvgReturn': community_data['ToplamGetiri'].mean(),
#                         'SuccessRate': community_data['Success'].mean()
#                     })

#         community_df = pd.DataFrame(community_analysis)

#         if len(community_df) > 0:
#             print(f"\nTopluluk Kalite Analizi:")
#             print(community_df[['Method', 'Size', 'InternalCorrelation', 'AvgReturn', 'SuccessRate']].round(3))

#         return communities_dict, community_df, G_community

#     def generate_summary_report(self):
#         """Özet rapor oluştur"""
#         if self.consolidated_data is None:
#             return "Veri yüklenmedi"

#         total_records = len(self.consolidated_data)
#         unique_brokers = self.consolidated_data['Broker'].nunique()
#         unique_stocks = self.consolidated_data['Hisse'].nunique()

#         buy_positions = len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'ALIŞ'])
#         sell_positions = len(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'SATIŞ'])

#         total_buy_volume = self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'ALIŞ']['ArdışıkToplam'].sum()
#         total_sell_volume = abs(self.consolidated_data[self.consolidated_data['İşlemYönü'] == 'SATIŞ']['ArdışıkToplam'].sum())

#         avg_return = self.consolidated_data['ToplamGetiri'].mean()

#         report = f"""
# {'='*60}
# ARACI KURUM NET POZİSYON ANALİZ RAPORU
# {'='*60}
# 📊 GENEL İSTATİSTİKLER:
# • Toplam Kayıt: {total_records:,}
# • Aracı Kurum Sayısı: {unique_brokers}
# • Hisse Sayısı: {unique_stocks}
# • Alış Pozisyonu: {buy_positions:,} (%{buy_positions/total_records*100:.1f})
# • Satış Pozisyonu: {sell_positions:,} (%{sell_positions/total_records*100:.1f})

# 💰 HACIM ANALİZİ:
# • Toplam Alış Hacmi: {total_buy_volume:,.0f} lot
# • Toplam Satış Hacmi: {total_sell_volume:,.0f} lot
# • Net Akış: {total_buy_volume - total_sell_volume:,.0f} lot
# • Ortalama Getiri: %{avg_return:.2f}

# 🎯 ANA BULGULAR:
# ✓ Broker strateji analizi tamamlandı
# ✓ Konsensüs analizi gerçekleştirildi
# ✓ Korelasyon matrisi oluşturuldu
# ✓ Başarı tahmin modeli geliştirildi
# ✓ Sürü davranışı analizi yapıldı
# ✓ Hafta içi pattern analizi tamamlandı
# ✓ Broker network analizi gerçekleştirildi
# ✓ Merkezi node analizi yapıldı
# ✓ Topluluk tespiti tamamlandı

# {'='*60}
#         """

#         return report

#     def run_comprehensive_analysis(self):
#         """Tüm analizleri çalıştır"""
#         print("🚀 ARACI KURUM NET POZİSYON ANALİZ SİSTEMİ")
#         print("="*50)

#         # Veri yükleme
#         if not self.load_broker_data():
#             print("❌ Veri yüklenemedi!")
#             return None

#         # Veri işleme
#         self.preprocess_data()

#         if self.consolidated_data is None or len(self.consolidated_data) == 0:
#             print("❌ Veri işlenemedi!")
#             return None

#         # Analizleri sırayla çalıştır
#         results = {}

#         try:
#             # Orijinal analizler
#             results['broker_strategies'] = self.broker_strategy_analysis()
#             results['smart_money'] = self.smart_money_analysis()
#             results['consecutive_success'] = self.consecutive_success_analysis()
#             results['anomaly_detection'] = self.anomaly_detection()
#             results['momentum_timing'] = self.momentum_timing_analysis()
#             results['sector_analysis'] = self.sector_analysis()
#             results['consensus_analysis'] = self.same_direction_consensus_analysis()
#             results['correlation_analysis'] = self.broker_correlation_analysis()
#             results['success_prediction'] = self.success_prediction_analysis()

#             # Yeni gelişmiş analizler
#             print("\n" + "="*50)
#             print("🔍 GELİŞMİŞ ANALİZLER BAŞLATILIYOR...")
#             print("="*50)

#             results['herd_behavior'] = self.herd_behavior_analysis()
#             results['weekend_effect'] = self.weekend_effect_analysis()
#             results['broker_networks'] = self.broker_network_analysis()
#             results['central_nodes'] = self.central_node_analysis()
#             results['communities'] = self.community_detection_analysis()

#             # Özet rapor
#             summary = self.generate_summary_report()
#             print(summary)

#             print("✅ TÜM ANALİZLER BAŞARIYLA TAMAMLANDI!")
#             print("\n🎯 YAPILAN ANALİZLER:")
#             print("✓ Broker strateji sınıflandırması")
#             print("✓ Smart money hareketleri")
#             print("✓ Ardışık işlem başarı analizi")
#             print("✓ Anomali tespiti")
#             print("✓ Momentum ve timing analizi")
#             print("✓ Sektör bazlı gruplandırma")
#             print("✓ Konsensüs ve karşıt görüş analizi")
#             print("✓ Broker korelasyon analizi")
#             print("✓ Machine learning başarı tahmini")
#             print("✓ Sürü davranışı analizi")
#             print("✓ Hafta içi pattern analizi")
#             print("✓ Broker network analizi")
#             print("✓ Merkezi node analizi")
#             print("✓ Topluluk tespiti")

#             return results

#         except Exception as e:
#             print(f"❌ Analiz hatası: {e}")
#             return None


In [ ]:
# # # KULLANIM

# if __name__ == "__main__":

#     # Excel dosya yolu (dinamik olarak bul)
#     # excel_files = glob.glob('HisseKurumSeriDurumlar_*.xlsx')
#     # if excel_files:
#     #     # Dosyaları tarihlerine göre sırala (dosya adındaki YYYYMMDD formatına göre)
#     #     excel_files.sort(key=lambda x: datetime.strptime(x.split('_')[-1].replace('.xlsx', ''), '%d%m%Y'), reverse=True)
#     #     excel_file_path = excel_files[0]
#     #     print(f"Bulunan en güncel dosya: {excel_file_path}")
#     # else:
#     #     print("HisseKurumSeriDurumlar_ ile başlayan Excel dosyası bulunamadı.")
#     #     excel_file_path = None
#     hissemaster_path, fiili_dolasim_dict = hisse_master_yukle()
#     if hissemaster_path:
#         # Analizci oluştur ve çalıştır
#         analyzer = BrokerNetPositionAnalyzer(hissemaster_path)
#         results = analyzer.run_comprehensive_analysis()

#         if results:
#             print("\n🎯 SONUÇ: Analiz başarıyla tamamlandı!")
#             print("📈 Yukarıdaki detaylı bulgulara göre aksiyon alabilirsiniz.")

#             # Önemli sonuçları özetleyelim
#             print("\n" + "="*50)
#             print("📊 ÖNE ÇIKAN BULGULAR:")
#             print("="*50)

#             # Sürü davranışı sonuçları
#             if 'herd_behavior' in results and results['herd_behavior'] is not None:
#                 strong_herd = results['herd_behavior'][results['herd_behavior']['HerdScore'] >= 0.8]
#                 if len(strong_herd) > 0:
#                     print(f"• {len(strong_herd)} hissede güçlü sürü davranışı tespit edildi")

#             # Merkezi broker sonuçları
#             if 'central_nodes' in results and len(results['central_nodes']) > 0:
#                 central_df = results['central_nodes'][0]  # İlk element central_df
#                 top_broker = central_df.iloc[0]['Broker']
#                 print(f"• En merkezi broker: {top_broker}")

#             # Smart money sonuçları
#             if 'smart_money' in results:
#                 smart_df = results['smart_money']
#                 top_smart_stock = smart_df.iloc[0]['Hisse']
#                 print(f"• En yüksek smart money skoru: {top_smart_stock}")
#         else:
#             print("\n❌ Analiz tamamlanamadı. Dosya yolu ve formatını kontrol edin.")
#     else:
#         print("\n❌ Analiz başlatılamadı çünkü geçerli bir Excel dosyası bulunamadı.")